# Nem a mirar bé les distribucions categoria per categoria, a veure si es solapen molt o k

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


# Put your notebook path here once (or set it via VS Code explorer)
NOTEBOOK_DIR = Path(r"C:\Users\labor\Documents\GitHub\TFM-Thermal-Walks\statistical_try")  # <-- change this

out_dir = NOTEBOOK_DIR / "figures"
out_dir.mkdir(parents=True, exist_ok=True)



df_votes = pd.read_csv(r"../../data/all_surveys(votes).csv")
print(df_votes.columns)



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tria aquí la variable que vols estudiar
# x_col = "<T-T_fixed+<T>>"
x_col = "<HDX-HDX_fixed+<HDX>>"

gender_col = "gender"
comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"

valid_genders = ["Woman", "Man"]

df = df_votes.copy()
df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
df = df.dropna(subset=[x_col]).copy()

if gender_col in df.columns:
    df = df[df[gender_col].isin(valid_genders)].copy()

def summarize_distribution(df_in, x_col, cat_col, gender_col=None):
    group_cols = [cat_col] if gender_col is None else [cat_col, gender_col]
    s = (
        df_in.groupby(group_cols, dropna=False)[x_col]
        .agg(
            n="count",
            mean="mean",
            median="median",
            std="std",
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
            min="min",
            max="max",
        )
        .reset_index()
    )
    s["iqr"] = s["q75"] - s["q25"]
    return s

In [ ]:
comfort_order = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

plot_df = df[df[comfort_col].isin(comfort_order)].copy()

fig, axes = plt.subplots(len(comfort_order), 1, figsize=(8, 2.2 * len(comfort_order)), sharex=True)

for ax, cat in zip(axes, comfort_order):
    sub = plot_df[plot_df[comfort_col] == cat]
    if sub.empty:
        ax.set_title(f"{cat} (no data)")
        continue

    for g in valid_genders:
        sub_g = sub[sub[gender_col] == g]
        if len(sub_g) == 0:
            continue
        ax.hist(sub_g[x_col], bins=30, alpha=0.5, density=True, label=f"{g} (n={len(sub_g)})")

    ax.set_title(cat)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel(x_col)
fig.suptitle(f"Distributions of {x_col} by thermal comfort", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tria aquí la variable que vols estudiar
# x_col = "<T-T_fixed+<T>>"
x_col = "<HDX-HDX_fixed+<HDX>>"

comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"

df = df_votes.copy()
df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
df = df.dropna(subset=[x_col]).copy()

def summarize_distribution(df_in, x_col, cat_col):
    s = (
        df_in.groupby(cat_col, dropna=False)[x_col]
        .agg(
            n="count",
            mean="mean",
            median="median",
            std="std",
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
            min="min",
            max="max",
        )
        .reset_index()
    )
    s["iqr"] = s["q75"] - s["q25"]
    return s

In [ ]:

comfort_order = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

plot_df = df[df[comfort_col].isin(comfort_order)].copy()

fig, axes = plt.subplots(
    len(comfort_order),
    1,
    figsize=(8, 2.2 * len(comfort_order)),
    sharex=True
)

for ax, cat in zip(axes, comfort_order):
    sub = plot_df[plot_df[comfort_col] == cat]

    if sub.empty:
        ax.set_title(f"{cat} (no data)")
        continue

    ax.hist(
        sub[x_col],
        bins=30,
        alpha=0.7,
        density=True,
        label=f"All votes (n={len(sub)})"
    )

    ax.set_title(cat)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel(x_col)
fig.suptitle(f"Global distributions of {x_col} by thermal comfort", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
sensation_order = [
    "Cool",
    "Slightly cool",
    "Neutral",
    "Slightly warm",
    "Warm",
    "Hot",
    "Very hot",
]

plot_df = df[df[sensation_col].isin(sensation_order)].copy()

fig, axes = plt.subplots(len(sensation_order), 1, figsize=(8, 2.2 * len(sensation_order)), sharex=True)

for ax, cat in zip(axes, sensation_order):
    sub = plot_df[plot_df[sensation_col] == cat]
    if sub.empty:
        ax.set_title(f"{cat} (no data)")
        continue

    for g in valid_genders:
        sub_g = sub[sub[gender_col] == g]
        if len(sub_g) == 0:
            continue
        ax.hist(sub_g[x_col], bins=35, alpha=0.5, density=True, label=f"{g} (n={len(sub_g)})")

    ax.set_title(cat)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel(x_col)
fig.suptitle(f"Distributions of {x_col} by thermal sensation", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df[df[comfort_col].isin(comfort_order)].copy()

data_w = [plot_df[(plot_df[comfort_col] == cat) & (plot_df[gender_col] == "Woman")][x_col].dropna().values for cat in comfort_order]
data_m = [plot_df[(plot_df[comfort_col] == cat) & (plot_df[gender_col] == "Man")][x_col].dropna().values for cat in comfort_order]

positions_w = np.arange(len(comfort_order)) * 2.0 - 0.18
positions_m = np.arange(len(comfort_order)) * 2.0 + 0.18

plt.figure(figsize=(12, 5))
plt.boxplot(data_w, positions=positions_w, widths=0.3, vert=True, patch_artist=False)
plt.boxplot(data_m, positions=positions_m, widths=0.3, vert=True, patch_artist=False)

plt.xticks(np.arange(len(comfort_order)) * 2.0, comfort_order, rotation=45, ha="right")
plt.ylabel(x_col)
plt.title(f"{x_col} distributions by thermal comfort and gender")
plt.grid(True, alpha=0.3)
plt.legend(["Woman", "Man"])
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df[df[sensation_col].isin(sensation_order)].copy()

data_w = [plot_df[(plot_df[sensation_col] == cat) & (plot_df[gender_col] == "Woman")][x_col].dropna().values for cat in sensation_order]
data_m = [plot_df[(plot_df[sensation_col] == cat) & (plot_df[gender_col] == "Man")][x_col].dropna().values for cat in sensation_order]

positions_w = np.arange(len(sensation_order)) * 2.0 - 0.18
positions_m = np.arange(len(sensation_order)) * 2.0 + 0.18

plt.figure(figsize=(12, 5))
plt.boxplot(data_w, positions=positions_w, widths=0.3, vert=True, patch_artist=False)
plt.boxplot(data_m, positions=positions_m, widths=0.3, vert=True, patch_artist=False)

plt.xticks(np.arange(len(sensation_order)) * 2.0, sensation_order, rotation=45, ha="right")
plt.ylabel(x_col)
plt.title(f"{x_col} distributions by thermal sensation and gender")
plt.grid(True, alpha=0.3)
plt.legend(["Woman", "Man"])
plt.tight_layout()
plt.show()

In [ ]:
comfort_summary = summarize_distribution(
    df[df[comfort_col].isin(comfort_order)],
    x_col=x_col,
    cat_col=comfort_col,
    gender_col=gender_col,
)
print(comfort_summary.sort_values([comfort_col, gender_col]))

In [ ]:
sensation_summary = summarize_distribution(
    df[df[sensation_col].isin(sensation_order)],
    x_col=x_col,
    cat_col=sensation_col,
    gender_col=gender_col,
)
print(sensation_summary.sort_values([sensation_col, gender_col]))

In [ ]:
from sklearn.metrics import roc_auc_score

def ecdf_vals(x):
    x = np.sort(np.asarray(x))
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

def histogram_overlap(x1, x2, bins=30):
    all_x = np.concatenate([x1, x2])
    edges = np.histogram_bin_edges(all_x, bins=bins)
    h1, _ = np.histogram(x1, bins=edges, density=True)
    h2, _ = np.histogram(x2, bins=edges, density=True)
    widths = np.diff(edges)
    overlap = np.sum(np.minimum(h1, h2) * widths)
    return overlap

def compare_two_categories(
    df_in,
    x_col,
    cat_col,
    cat_a,
    cat_b,
    gender=None,
    gender_col="gender",
    bins=25
):
    sub = df_in[df_in[cat_col].isin([cat_a, cat_b])].copy()

    if gender is not None:
        sub = sub[sub[gender_col] == gender].copy()

    x_a = sub.loc[sub[cat_col] == cat_a, x_col].dropna().to_numpy()
    x_b = sub.loc[sub[cat_col] == cat_b, x_col].dropna().to_numpy()

    if len(x_a) == 0 or len(x_b) == 0:
        print("One of the categories has no data.")
        return

    # AUC
    y = np.r_[np.zeros(len(x_a)), np.ones(len(x_b))]
    x = np.r_[x_a, x_b]
    auc = roc_auc_score(y, x)

    # summary
    summary = pd.DataFrame({
        "category": [cat_a, cat_b],
        "n": [len(x_a), len(x_b)],
        "mean": [x_a.mean(), x_b.mean()],
        "median": [np.median(x_a), np.median(x_b)],
        "std": [x_a.std(ddof=1), x_b.std(ddof=1)],
        "q25": [np.quantile(x_a, 0.25), np.quantile(x_b, 0.25)],
        "q75": [np.quantile(x_a, 0.75), np.quantile(x_b, 0.75)],
    })
    summary["iqr"] = summary["q75"] - summary["q25"]

    overlap = histogram_overlap(x_a, x_b, bins=bins)

    print(f"\n=== Comparison: {cat_a} vs {cat_b}" + (f" | {gender}" if gender else "") + " ===")
    print(summary)
    print(f"\nAUC ({cat_a}=0, {cat_b}=1): {auc:.3f}")
    print(f"Symmetric overlap from histograms (~1 = very similar): {overlap:.3f}")
    print(f"Difference in means ({cat_b} - {cat_a}): {x_b.mean() - x_a.mean():.3f}")
    print(f"Difference in medians ({cat_b} - {cat_a}): {np.median(x_b) - np.median(x_a):.3f}")

    # plots
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # histogram
    axes[0].hist(x_a, bins=bins, alpha=0.5, density=True, label=f"{cat_a} (n={len(x_a)})")
    axes[0].hist(x_b, bins=bins, alpha=0.5, density=True, label=f"{cat_b} (n={len(x_b)})")
    axes[0].set_title("Histogram")
    axes[0].set_xlabel(x_col)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # ECDF
    xa_s, ya = ecdf_vals(x_a)
    xb_s, yb = ecdf_vals(x_b)
    axes[1].plot(xa_s, ya, label=cat_a)
    axes[1].plot(xb_s, yb, label=cat_b)
    axes[1].set_title("ECDF")
    axes[1].set_xlabel(x_col)
    axes[1].set_ylabel("Cumulative probability")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.suptitle(f"{cat_a} vs {cat_b}" + (f" | {gender}" if gender else ""))
    plt.tight_layout()
    plt.show()

In [ ]:
compare_two_categories(
    df_in=df,
    x_col=x_col,
    cat_col=sensation_col,
    cat_a="Hot",
    cat_b="Very hot",
    gender=None,
)

In [ ]:
compare_two_categories(
    df_in=df,
    x_col=x_col,
    cat_col=sensation_col,
    cat_a="Cool",
    cat_b="Slightly cool",
    gender=None,
)

compare_two_categories(
    df_in=df,
    x_col=x_col,
    cat_col=sensation_col,
    cat_a="Hot",
    cat_b="Very hot",
    gender="Man",
)

In [ ]:
compare_two_categories(
    df_in=df,
    x_col=x_col,
    cat_col=comfort_col,
    cat_a="Uncomfortable",
    cat_b="Very uncomfortable",
    gender=None,
)

In [ ]:
compare_two_categories(
    df_in=df,
    x_col=x_col,
    cat_col=comfort_col,
    cat_a="Comfortable",
    cat_b="Very comfortable",
    gender=None,
)

compare_two_categories(
    df_in=df,
    x_col=x_col,
    cat_col=comfort_col,
    cat_a="Comfortable",
    cat_b="Very comfortable",
    gender="Man",
)

In [ ]:
compare_two_categories(
    df_in=df,
    x_col=x_col,
    cat_col=sensation_col,
    cat_a="Slightly cool",
    cat_b="Cool",
    gender=None,
)

# Proba extrana

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_category_hdx_heatmap(
    df_in,
    x_col,
    cat_col,
    categories,
    bins=25,
    min_n=10,
    title=None,
):
    sub = df_in[df_in[cat_col].isin(categories)].copy()
    sub[x_col] = pd.to_numeric(sub[x_col], errors="coerce")
    sub = sub.dropna(subset=[x_col, cat_col]).copy()

    edges = np.histogram_bin_edges(sub[x_col], bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])

    rows = []
    labels = []

    for cat in categories:
        x = sub.loc[sub[cat_col] == cat, x_col].dropna().to_numpy()

        if len(x) < min_n:
            continue

        counts, _ = np.histogram(x, bins=edges)

        # Row-normalized: each row is P(HDX bin | category)
        row = counts / counts.sum()

        rows.append(row)
        labels.append(f"{cat} (n={len(x)})")

    mat = np.vstack(rows)

    plt.figure(figsize=(10, 5))
    plt.imshow(
        mat,
        aspect="auto",
        interpolation="nearest",
        extent=[centers.min(), centers.max(), len(labels), 0],
    )

    plt.colorbar(label="P(HDX bin | category)")
    plt.yticks(np.arange(len(labels)) + 0.5, labels)
    plt.xlabel(x_col)
    plt.ylabel("Category")

    if title is None:
        title = f"Category-specific HDX behavior: {cat_col}"

    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_category_hdx_heatmap(
    df_in=df,
    x_col=x_col,
    cat_col=comfort_col,
    categories=comfort_order,
    bins=25,
    title="Thermal comfort categories as HDX distributions"
)

plot_category_hdx_heatmap(
    df_in=df,
    x_col=x_col,
    cat_col=sensation_col,
    categories=sensation_order,
    bins=25,
    title="Thermal sensation categories as HDX distributions"
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

x_col = "<HDX-HDX_fixed+<HDX>>"
comfort_col = "thermal_comfort"

comfort_order = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

min_bin_n = 10
bins = 25

df_h = df_votes.copy()
df_h[x_col] = pd.to_numeric(df_h[x_col], errors="coerce")
df_h = df_h[df_h[comfort_col].isin(comfort_order)].dropna(subset=[x_col, comfort_col]).copy()

edges = np.histogram_bin_edges(df_h[x_col], bins=bins)
centers = 0.5 * (edges[:-1] + edges[1:])

df_h["HDX_bin"] = pd.cut(df_h[x_col], bins=edges, include_lowest=True)

# Counts: rows = comfort category, columns = HDX bins
tab = pd.crosstab(df_h[comfort_col], df_h["HDX_bin"])
tab = tab.reindex(comfort_order)

# Total observations per HDX bin
bin_counts = tab.sum(axis=0)

# Column-normalized: P(category | HDX bin)
tab_col = tab.div(bin_counts, axis=1)

# Mask bins with fewer than min_bin_n observations
tab_col_masked = tab_col.copy()
tab_col_masked.loc[:, bin_counts < min_bin_n] = np.nan

# Plot
cmap = plt.cm.viridis.copy()
cmap.set_bad(color="white")

plt.figure(figsize=(10, 5))

plt.imshow(
    tab_col_masked.values,
    aspect="auto",
    interpolation="nearest",
    cmap=cmap,
    extent=[centers.min(), centers.max(), len(comfort_order), 0],
)

plt.colorbar(label="P(category | HDX bin)")
plt.yticks(np.arange(len(comfort_order)) + 0.5, comfort_order)
plt.xlabel(x_col)
plt.ylabel("Comfort category")
plt.title(f"Comfort response distribution by HDX bin\nBins with n < {min_bin_n} masked")

plt.tight_layout()
plt.show()

print("HDX bin counts:")
print(bin_counts)

bin_counts = tab.sum(axis=0)
print(bin_counts)

# estimating a changing point

In [ ]:
def ecdf_xy(x):
    x = np.sort(np.asarray(x))
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y


def fit_piecewise_ecdf(x, n_breaks=100, q_low=0.15, q_high=0.85):
    x, y = ecdf_xy(x)

    if len(x) < 20:
        return None

    candidates = np.linspace(
        np.quantile(x, q_low),
        np.quantile(x, q_high),
        n_breaks
    )

    best = None

    for bp in candidates:
        hinge = np.maximum(0, x - bp)

        X = np.column_stack([
            np.ones_like(x),
            x,
            hinge,
        ])

        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        yhat = X @ beta

        sse = np.sum((y - yhat) ** 2)

        slope_low = beta[1]
        slope_high = beta[1] + beta[2]

        if best is None or sse < best["sse"]:
            best = {
                "breakpoint": bp,
                "slope_low": slope_low,
                "slope_high": slope_high,
                "slope_change": slope_high - slope_low,
                "slope_ratio_high_over_low": slope_high / slope_low if slope_low != 0 else np.nan,
                "sse": sse,
                "beta": beta,
                "x": x,
                "y": y,
                "yhat": yhat,
            }

    return best


def category_piecewise_ecdf_table(
    df_in,
    x_col,
    cat_col,
    categories,
    min_n=25,
):
    rows = []
    fits = {}

    for cat in categories:
        x = (
            df_in.loc[df_in[cat_col] == cat, x_col]
            .pipe(pd.to_numeric, errors="coerce")
            .dropna()
            .to_numpy()
        )

        if len(x) < min_n:
            continue

        fit = fit_piecewise_ecdf(x)

        if fit is None:
            continue

        fits[cat] = fit

        rows.append({
            "category": cat,
            "n": len(x),
            "breakpoint": fit["breakpoint"],
            "slope_low_region": fit["slope_low"],
            "slope_high_region": fit["slope_high"],
            "slope_change_high_minus_low": fit["slope_change"],
            "slope_ratio_high_over_low": fit["slope_ratio_high_over_low"],
        })

    table = pd.DataFrame(rows)
    return table, fits

In [ ]:
comfort_piecewise_table, comfort_piecewise_fits = category_piecewise_ecdf_table(
    df_in=df,
    x_col=x_col,
    cat_col=comfort_col,
    categories=comfort_order,
    min_n=25,
)

print(comfort_piecewise_table.round(3))

In [ ]:
sensation_piecewise_table, sensation_piecewise_fits = category_piecewise_ecdf_table(
    df_in=df,
    x_col=x_col,
    cat_col=sensation_col,
    categories=sensation_order,
    min_n=25,
)

print(sensation_piecewise_table.round(3))

In [ ]:
def plot_piecewise_ecdf_fit(fits, category, x_col):
    fit = fits[category]

    plt.figure(figsize=(8, 5))

    plt.plot(
        fit["x"],
        fit["y"],
        label="Observed ECDF",
    )

    plt.plot(
        fit["x"],
        fit["yhat"],
        linestyle="--",
        label="Piecewise ECDF fit",
    )

    plt.axvline(
        fit["breakpoint"],
        linestyle=":",
        label=f"Change point ≈ {fit['breakpoint']:.2f}",
    )

    plt.xlabel(x_col)
    plt.ylabel("Cumulative probability")
    plt.title(f"{category}: ECDF slope change")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_piecewise_ecdf_fit(
    comfort_piecewise_fits,
    category="Comfortable",
    x_col=x_col,
)

plot_piecewise_ecdf_fit(
    comfort_piecewise_fits,
    category="Very uncomfortable",
    x_col=x_col,
)

plot_piecewise_ecdf_fit(
    sensation_piecewise_fits,
    category="Hot",
    x_col=x_col,
)

# ja que ara no esta soortint, probarem més coses

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

def estimate_pooled_hdx_valley(
    df_in,
    x_col,
    search_range=(35, 39),
    bw_method=0.25,
    plot=True,
):
    x = pd.to_numeric(df_in[x_col], errors="coerce").dropna().to_numpy()

    kde = gaussian_kde(x, bw_method=bw_method)

    grid = np.linspace(np.quantile(x, 0.01), np.quantile(x, 0.99), 1000)
    density = kde(grid)

    mask = (grid >= search_range[0]) & (grid <= search_range[1])
    boundary = grid[mask][np.argmin(density[mask])]

    if plot:
        plt.figure(figsize=(8, 5))
        plt.plot(grid, density, label="Pooled HDX density")
        plt.axvline(boundary, linestyle="--", label=f"Valley ≈ {boundary:.2f}")
        plt.axvspan(search_range[0], search_range[1], alpha=0.15, label="Search range")
        plt.xlabel(x_col)
        plt.ylabel("Density")
        plt.title("Pooled HDX density: estimated middle/high regime valley")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    return boundary

boundary = estimate_pooled_hdx_valley(
    df_in=df,
    x_col=x_col,
    search_range=(35, 39),
    bw_method=0.25,
)

print("Estimated middle/high HDX boundary:", boundary)

In [ ]:
def category_relative_enrichment(
    df_in,
    x_col,
    cat_col,
    categories,
    bins=25,
    alpha=0.5,
    min_n=20,
):
    sub = df_in[df_in[cat_col].isin(categories)].copy()
    sub[x_col] = pd.to_numeric(sub[x_col], errors="coerce")
    sub = sub.dropna(subset=[x_col, cat_col]).copy()

    edges = np.histogram_bin_edges(sub[x_col], bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])

    pooled_counts, _ = np.histogram(sub[x_col], bins=edges)

    # Background probability P(HDX bin)
    pooled_prob = (pooled_counts + alpha) / (pooled_counts.sum() + alpha * len(pooled_counts))

    rows = []

    for cat in categories:
        x_cat = sub.loc[sub[cat_col] == cat, x_col].dropna().to_numpy()

        if len(x_cat) < min_n:
            continue

        cat_counts, _ = np.histogram(x_cat, bins=edges)

        # Category probability P(HDX bin | category)
        cat_prob = (cat_counts + alpha) / (cat_counts.sum() + alpha * len(cat_counts))

        # Relative enrichment:
        # > 1 means category is overrepresented in that HDX bin
        # < 1 means category is underrepresented
        enrichment = cat_prob / pooled_prob

        for c, e, cp, pp, count in zip(centers, enrichment, cat_prob, pooled_prob, cat_counts):
            rows.append({
                "category": cat,
                "HDX_center": c,
                "category_count": count,
                "P_bin_given_category": cp,
                "P_bin_pooled": pp,
                "relative_enrichment": e,
                "log_relative_enrichment": np.log(e),
                "n_category": len(x_cat),
            })

    return pd.DataFrame(rows)

In [ ]:
comfort_enrich = category_relative_enrichment(
    df_in=df,
    x_col=x_col,
    cat_col=comfort_col,
    categories=comfort_order,
    bins=25,
)

print(comfort_enrich.head())

In [ ]:
sensation_enrich = category_relative_enrichment(
    df_in=df,
    x_col=x_col,
    cat_col=sensation_col,
    categories=sensation_order,
    bins=25,
)

print(sensation_enrich.head())

In [ ]:
def plot_enrichment_curves(enrich_df, categories, boundary=None, title=None):
    plt.figure(figsize=(10, 5))

    for cat in categories:
        g = enrich_df[enrich_df["category"] == cat]

        if g.empty:
            continue

        plt.plot(
            g["HDX_center"],
            g["relative_enrichment"],
            marker="o",
            label=cat,
        )

    plt.axhline(1, linestyle="--", label="Same as pooled background")

    if boundary is not None:
        plt.axvline(boundary, linestyle=":", label=f"Boundary ≈ {boundary:.2f}")

    plt.xlabel("HDX")
    plt.ylabel("Relative enrichment")
    plt.title(title or "Category enrichment relative to pooled HDX distribution")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_enrichment_curves(
    comfort_enrich,
    categories=comfort_order,
    boundary=boundary,
    title="Comfort categories: behavior relative to pooled HDX structure",
)

# summary

In [ ]:
def regime_enrichment_table(
    df_in,
    x_col,
    cat_col,
    categories,
    boundary,
):
    sub = df_in[df_in[cat_col].isin(categories)].copy()
    sub[x_col] = pd.to_numeric(sub[x_col], errors="coerce")
    sub = sub.dropna(subset=[x_col, cat_col]).copy()

    sub["HDX_regime"] = np.where(
        sub[x_col] <= boundary,
        "Regime 1: lower/middle HDX",
        "Regime 2: higher HDX",
    )

    # P(regime | all data)
    pooled_regime_prob = sub["HDX_regime"].value_counts(normalize=True)

    rows = []

    for cat in categories:
        g = sub[sub[cat_col] == cat]

        if len(g) == 0:
            continue

        cat_regime_prob = g["HDX_regime"].value_counts(normalize=True)

        for regime in pooled_regime_prob.index:
            p_cat = cat_regime_prob.get(regime, 0)
            p_pool = pooled_regime_prob.get(regime, np.nan)

            rows.append({
                "category": cat,
                "n": len(g),
                "regime": regime,
                "P_regime_given_category": p_cat,
                "P_regime_pooled": p_pool,
                "relative_enrichment": p_cat / p_pool,
            })

    return pd.DataFrame(rows)

comfort_regime_enrich = regime_enrichment_table(
    df_in=df,
    x_col=x_col,
    cat_col=comfort_col,
    categories=comfort_order,
    boundary=boundary,
)

print(comfort_regime_enrich.round(3))

In [ ]:
import numpy as np
import pandas as pd

# ----------------------------
# Columns
# ----------------------------
x_col = "<HDX-HDX_fixed+<HDX>>"
comfort_col = "thermal_comfort"

# Use the same boundary you used before
# Example only — replace with your actual estimated boundary
boundary = 37.5  

# ----------------------------
# Create dataframe with regimes
# ----------------------------
df_reg = df_votes.copy()

df_reg[x_col] = pd.to_numeric(df_reg[x_col], errors="coerce")
df_reg = df_reg.dropna(subset=[x_col, comfort_col]).copy()

df_reg["HDX_regime"] = np.where(
    df_reg[x_col] <= boundary,
    "Regime 1: lower/middle HDX",
    "Regime 2: higher HDX"
)

# ----------------------------
# Internal behavior of each comfort category inside each regime
# ----------------------------
internal_behavior = (
    df_reg
    .groupby([comfort_col, "HDX_regime"], observed=False)
    .agg(
        n=(x_col, "count"),
        mean_HDX=(x_col, "mean"),
        median_HDX=(x_col, "median"),
        std_HDX=(x_col, "std"),
        q25_HDX=(x_col, lambda x: x.quantile(0.25)),
        q75_HDX=(x_col, lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

internal_behavior["iqr_HDX"] = (
    internal_behavior["q75_HDX"] - internal_behavior["q25_HDX"]
)

print(internal_behavior.round(3))


comfort_order = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

internal_behavior[comfort_col] = pd.Categorical(
    internal_behavior[comfort_col],
    categories=comfort_order,
    ordered=True
)

internal_behavior = internal_behavior.sort_values([comfort_col, "HDX_regime"])

print(internal_behavior.round(3))

In [ ]:
import scipy.stats as stats

# Regime 1 only
r1 = df_reg[df_reg["HDX_regime"] == "Regime 1: lower/middle HDX"]

groups_r1 = [
    g[x_col].dropna().values
    for _, g in r1.groupby("thermal_comfort", observed=False)
    if len(g) > 5
]

kw_r1 = stats.kruskal(*groups_r1)
print("Regime 1 Kruskal-Wallis:", kw_r1)

# Regime 2 only
r2 = df_reg[df_reg["HDX_regime"] == "Regime 2: higher HDX"]

groups_r2 = [
    g[x_col].dropna().values
    for _, g in r2.groupby("thermal_comfort", observed=False)
    if len(g) > 5
]

kw_r2 = stats.kruskal(*groups_r2)
print("Regime 2 Kruskal-Wallis:", kw_r2)

# tests to check if we are correct

In [ ]:
import numpy as np
import pandas as pd

x_col = "<HDX-HDX_fixed+<HDX>>"
comfort_col = "thermal_comfort"
walk_col = "walk"

boundary = 37.5  # replace with your current boundary

df_check = df_votes.copy()

df_check[x_col] = pd.to_numeric(df_check[x_col], errors="coerce")
df_check["walk"] = df_check["space_code"].astype(str).str[:2]

df_check = df_check.dropna(subset=[x_col, comfort_col, walk_col]).copy()

df_check["HDX_regime"] = np.where(
    df_check[x_col] <= boundary,
    "Regime 1",
    "Regime 2"
)

walk_regime_counts = pd.crosstab(
    df_check[walk_col],
    df_check["HDX_regime"]
)

walk_regime_percent_within_regime = (
    walk_regime_counts
    / walk_regime_counts.sum(axis=0)
    * 100
)

print("Counts by walk and regime:")
print(walk_regime_counts)

print("\nPercentage contribution of each walk to each regime:")
print(walk_regime_percent_within_regime.round(1))

print("\nTop walks contributing to Regime 2:")
print(
    walk_regime_percent_within_regime["Regime 2"]
    .sort_values(ascending=False)
    .head(20)
)

### trobat k al regim 2 hi contribueixen el 33% dels walks

In [ ]:
from scipy import stats

comfort_order = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

def kruskal_by_regime_for_boundary(df_in, boundary):
    d = df_in.copy()
    d["HDX_regime"] = np.where(d[x_col] <= boundary, "Regime 1", "Regime 2")
    d["TCV"] = d[comfort_col].map(tcv_map)

    rows = []

    for regime in ["Regime 1", "Regime 2"]:
        r = d[d["HDX_regime"] == regime].copy()

        groups = []
        for cat in comfort_order:
            vals = r.loc[r[comfort_col] == cat, x_col].dropna().values
            if len(vals) >= 5:
                groups.append(vals)

        if len(groups) < 2:
            continue

        H, p = stats.kruskal(*groups)

        N = sum(len(g) for g in groups)
        k = len(groups)
        epsilon2 = (H - k + 1) / (N - k) if N > k else np.nan

        rho, rho_p = stats.spearmanr(
            r["TCV"],
            r[x_col],
            nan_policy="omit"
        )

        cat_means = (
            r.groupby(comfort_col)[x_col]
            .mean()
            .reindex(comfort_order)
            .dropna()
        )

        mean_spread = cat_means.max() - cat_means.min()

        rows.append({
            "boundary": boundary,
            "regime": regime,
            "N": N,
            "H": H,
            "p": p,
            "epsilon2": epsilon2,
            "spearman_rho_TCV_HDX": rho,
            "spearman_p": rho_p,
            "category_mean_spread": mean_spread,
        })

    return rows


df_base = df_votes.copy()
df_base[x_col] = pd.to_numeric(df_base[x_col], errors="coerce")
df_base = df_base[df_base[comfort_col].isin(comfort_order)].copy()
df_base = df_base.dropna(subset=[x_col, comfort_col]).copy()

boundaries = np.linspace(35, 39, 41)

rows = []
for b in boundaries:
    rows.extend(kruskal_by_regime_for_boundary(df_base, b))

boundary_sensitivity = pd.DataFrame(rows)
boundary_sensitivity.to_csv("csvs/boundary_sensitivity_analysis.csv", index=False)

print(boundary_sensitivity.round(4))

## sembla que el rang 37.3-39 s'aguanta això + plot

In [ ]:
import matplotlib.pyplot as plt

for metric in ["p", "epsilon2", "spearman_rho_TCV_HDX", "category_mean_spread"]:
    plt.figure(figsize=(8, 5))

    for regime in ["Regime 1", "Regime 2"]:
        g = boundary_sensitivity[boundary_sensitivity["regime"] == regime]
        plt.plot(g["boundary"], g[metric], marker="o", label=regime)

    if metric == "p":
        plt.axhline(0.05, linestyle="--", label="p = 0.05")

    plt.xlabel("HDX boundary")
    plt.ylabel(metric)
    plt.title(f"Boundary sensitivity: {metric}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## No estic molt segur de les ultimes mètriques però el p-value i la epsilon 2 semblen molt clares

In [ ]:
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt

def estimate_hdx_valley(df_in, x_col, search_range=(35, 39), bw_method=0.25):
    x = pd.to_numeric(df_in[x_col], errors="coerce").dropna().to_numpy()

    kde = gaussian_kde(x, bw_method=bw_method)

    grid = np.linspace(np.percentile(x, 1), np.percentile(x, 99), 1000)
    density = kde(grid)

    mask = (grid >= search_range[0]) & (grid <= search_range[1])

    boundary = grid[mask][np.argmin(density[mask])]

    plt.figure(figsize=(8, 5))
    plt.plot(grid, density, label="Pooled HDX density")
    plt.axvline(boundary, linestyle="--", label=f"Estimated valley = {boundary:.2f}")
    plt.axvspan(search_range[0], search_range[1], alpha=0.15, label="Search range")
    plt.xlabel("HDX")
    plt.ylabel("Density")
    plt.title("Pooled HDX density and estimated regime boundary")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return boundary

boundary_estimated = estimate_hdx_valley(
    df_base,
    x_col=x_col,
    search_range=(35, 39),
    bw_method=0.25
)

print("Estimated HDX regime boundary:", boundary_estimated)

In [ ]:
for bw in [0.15, 0.20, 0.25, 0.30, 0.40]:
    b = estimate_hdx_valley(
        df_base,
        x_col=x_col,
        search_range=(35, 39),
        bw_method=bw
    )
    print("bandwidth:", bw, "boundary:", b)

# sembla que el boundary es queda apx cap al mateix lloc

# wierd one-walk out boostrap

In [ ]:
def compute_regime_summary(df_in, boundary):
    d = df_in.copy()
    d["HDX_regime"] = np.where(d[x_col] <= boundary, "Regime 1", "Regime 2")
    d["TCV"] = d[comfort_col].map(tcv_map)

    out = {}

    for regime in ["Regime 1", "Regime 2"]:
        r = d[d["HDX_regime"] == regime].dropna(subset=["TCV", x_col]).copy()

        groups = []
        for cat in comfort_order:
            vals = r.loc[r[comfort_col] == cat, x_col].dropna().values
            if len(vals) >= 5:
                groups.append(vals)

        if len(groups) >= 2:
            H, p = stats.kruskal(*groups)
        else:
            H, p = np.nan, np.nan

        rho, rho_p = stats.spearmanr(r["TCV"], r[x_col], nan_policy="omit")

        means = (
            r.groupby(comfort_col)[x_col]
            .mean()
            .reindex(comfort_order)
            .dropna()
        )

        out[f"{regime}_N"] = len(r)
        out[f"{regime}_H"] = H
        out[f"{regime}_p"] = p
        out[f"{regime}_rho"] = rho
        out[f"{regime}_mean_spread"] = means.max() - means.min()

    return out


df_walk = df_base.copy()
df_walk["walk"] = df_votes.loc[df_walk.index, "space_code"].astype(str).str[:2]

rows = []

for w in sorted(df_walk["walk"].dropna().unique()):
    sub = df_walk[df_walk["walk"] != w].copy()
    res = compute_regime_summary(sub, boundary=boundary)
    res["walk_removed"] = w
    rows.append(res)

loo = pd.DataFrame(rows)

print(loo.round(4))

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loo["walk_removed"], loo["Regime 1_rho"], marker="o", label="Regime 1")
plt.plot(loo["walk_removed"], loo["Regime 2_rho"], marker="o", label="Regime 2")
plt.axhline(0, linestyle="--")
plt.xlabel("Walk removed")
plt.ylabel("Spearman rho: comfort order vs HDX")
plt.title("Leave-one-walk-out robustness")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# sembla ok, fem cluster boosrap by walk

In [ ]:
def cluster_bootstrap_by_walk(df_in, boundary, n_boot=1000, random_state=0):
    rng = np.random.default_rng(random_state)

    walks = df_in["walk"].dropna().unique()
    rows = []

    for i in range(n_boot):
        sampled_walks = rng.choice(walks, size=len(walks), replace=True)

        pieces = []
        for j, w in enumerate(sampled_walks):
            g = df_in[df_in["walk"] == w].copy()
            g["boot_walk"] = f"{w}_{j}"
            pieces.append(g)

        boot = pd.concat(pieces, ignore_index=True)

        res = compute_regime_summary(boot, boundary)
        res["boot"] = i
        rows.append(res)

    return pd.DataFrame(rows)


boot = cluster_bootstrap_by_walk(
    df_walk,
    boundary=boundary,
    n_boot=1000,
    random_state=1
)

for col in ["Regime 1_rho", "Regime 2_rho", "Regime 1_mean_spread", "Regime 2_mean_spread"]:
    vals = boot[col].dropna()
    print(
        col,
        "median =", np.nanmedian(vals),
        "95% CI =", np.nanpercentile(vals, [2.5, 97.5])
    )

boot["rho_difference_R1_minus_R2"] = boot["Regime 1_rho"] - boot["Regime 2_rho"]

print(
    "rho difference R1 - R2:",
    "median =", np.nanmedian(boot["rho_difference_R1_minus_R2"]),
    "95% CI =", np.nanpercentile(boot["rho_difference_R1_minus_R2"].dropna(), [2.5, 97.5])
)

In [ ]:
def regime_rho_difference(df_in, boundary, y_col="TCV"):
    d = df_in.copy()
    d["HDX_regime"] = np.where(d[x_col] <= boundary, "Regime 1", "Regime 2")

    rhos = {}

    for regime in ["Regime 1", "Regime 2"]:
        r = d[d["HDX_regime"] == regime].dropna(subset=[x_col, y_col])
        if len(r) < 10:
            rhos[regime] = np.nan
        else:
            rho, _ = stats.spearmanr(r[y_col], r[x_col])
            rhos[regime] = rho

    return rhos["Regime 1"] - rhos["Regime 2"]


df_perm = df_walk.copy()
df_perm["TCV"] = df_perm[comfort_col].map(tcv_map)
df_perm = df_perm.dropna(subset=[x_col, "TCV", "walk"]).copy()

obs_stat = regime_rho_difference(df_perm, boundary)

rng = np.random.default_rng(0)
perm_stats = []

n_perm = 2000

for i in range(n_perm):
    d = df_perm.copy()

    # Shuffle comfort scores within each walk
    d["TCV_perm"] = (
        d.groupby("walk")["TCV"]
        .transform(lambda s: rng.permutation(s.to_numpy()))
    )

    stat = regime_rho_difference(d, boundary, y_col="TCV_perm")
    perm_stats.append(stat)

perm_stats = np.array(perm_stats)

p_perm = np.mean(np.abs(perm_stats) >= np.abs(obs_stat))

print("Observed rho difference R1 - R2:", obs_stat)
print("Within-walk permutation p-value:", p_perm)

plt.figure(figsize=(8, 5))
plt.hist(perm_stats, bins=40, alpha=0.7)
plt.axvline(obs_stat, linestyle="--", label=f"Observed = {obs_stat:.3f}")
plt.axvline(-obs_stat, linestyle="--")
plt.xlabel("Permuted rho difference: Regime 1 - Regime 2")
plt.ylabel("Count")
plt.title("Within-walk permutation test")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# sembal que sí que es significant la permuted rho difference

In [ ]:
from scipy.stats import levene

d = df_walk.copy()
d["HDX_regime"] = np.where(d[x_col] <= boundary, "Regime 1", "Regime 2")

for cat in comfort_order:
    r1 = d.loc[
        (d[comfort_col] == cat) & (d["HDX_regime"] == "Regime 1"),
        x_col
    ].dropna()

    r2 = d.loc[
        (d[comfort_col] == cat) & (d["HDX_regime"] == "Regime 2"),
        x_col
    ].dropna()

    if len(r1) >= 10 and len(r2) >= 10:
        stat, p = levene(r1, r2, center="median")
        print(
            cat,
            "n1 =", len(r1),
            "n2 =", len(r2),
            "IQR1 =", r1.quantile(0.75) - r1.quantile(0.25),
            "IQR2 =", r2.quantile(0.75) - r2.quantile(0.25),
            "Levene p =", p
        )
        

# no pillo massa aixo

In [ ]:
import statsmodels.formula.api as smf

d = df_walk.copy()
d["TCV"] = d[comfort_col].map(tcv_map)
d = d.dropna(subset=[x_col, "TCV", "walk"]).copy()

d["high_regime"] = (d[x_col] > boundary).astype(int)
d["HDX_centered"] = d[x_col] - boundary

# Cluster-robust standard errors by walk
model = smf.ols(
    "TCV ~ HDX_centered * high_regime",
    data=d
).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["walk"]}
)

print(model.summary())

In [ ]:
# Optional controls
d["sun_num"] = d["sun"].map({
    "In full shade": -1,
    "In a mixture of sun and shadow": 0,
    "In full sun": 1,
})

d["wind_num"] = d["wind"].map({
    "A strong wind": -1,
    "A moderate wind": -1,
    "A light wind": 0,
    "A very light wind": 0,
    "It's not windy": 1,
})

model_controls = smf.ols(
    "TCV ~ HDX_centered * high_regime + C(gender) + sun_num + wind_num",
    data=d
).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["walk"]}
)

print(model_controls.summary())

In [ ]:
import pandas as pd

tables_to_export = {}

possible_tables = [
    "walk_regime_counts",
    "walk_regime_percent_within_regime",
    "boundary_sensitivity",
    "loo",
    "boot",
    "comfort_regime_enrich",
    "sensation_regime_enrich",
    "internal_behavior",
    "sensation_internal_behavior",
]

for name in possible_tables:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame):
            tables_to_export[name] = obj
        elif isinstance(obj, pd.Series):
            tables_to_export[name] = obj.to_frame()

with pd.ExcelWriter("HDX_regime_results.xlsx") as writer:
    for name, table in tables_to_export.items():
        table.to_excel(writer, sheet_name=name[:31], index=True)

print("Exported:", list(tables_to_export.keys()))

# No tot ha quedat clar així que probem un test més

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------
# Config
# ----------------------------
x_col = "<HDX-HDX_fixed+<HDX>>"
comfort_col = "thermal_comfort"
walk_col = "walk"

boundary = 37.5  # use your chosen/estimated boundary

comfort_order = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

# ----------------------------
# Prepare data
# ----------------------------
df_test = df_votes.copy()

df_test[x_col] = pd.to_numeric(df_test[x_col], errors="coerce")
df_test["walk"] = df_test["space_code"].astype(str).str[:2]
df_test["TCV"] = df_test[comfort_col].map(tcv_map)

df_test = df_test.dropna(subset=[x_col, comfort_col, "walk", "TCV"]).copy()

df_test["HDX_regime"] = np.where(
    df_test[x_col] <= boundary,
    "Regime 1",
    "Regime 2",
)

# ----------------------------
# Identify mixed walks
# ----------------------------
walk_regime_counts = pd.crosstab(
    df_test["walk"],
    df_test["HDX_regime"]
)

mixed_walks = walk_regime_counts[
    (walk_regime_counts.get("Regime 1", 0) > 0)
    & (walk_regime_counts.get("Regime 2", 0) > 0)
].index

df_mixed = df_test[df_test["walk"].isin(mixed_walks)].copy()

print("Total walks:", df_test["walk"].nunique())
print("Mixed walks:", len(mixed_walks))
print("Mixed-walk observations:", len(df_mixed))
print(walk_regime_counts.loc[mixed_walks])

In [ ]:
def internal_behavior_table(df_in):
    out = (
        df_in
        .groupby([comfort_col, "HDX_regime"], observed=False)
        .agg(
            n=(x_col, "count"),
            mean_HDX=(x_col, "mean"),
            median_HDX=(x_col, "median"),
            std_HDX=(x_col, "std"),
            q25_HDX=(x_col, lambda x: x.quantile(0.25)),
            q75_HDX=(x_col, lambda x: x.quantile(0.75)),
        )
        .reset_index()
    )

    out["iqr_HDX"] = out["q75_HDX"] - out["q25_HDX"]

    out[comfort_col] = pd.Categorical(
        out[comfort_col],
        categories=comfort_order,
        ordered=True,
    )

    return out.sort_values([comfort_col, "HDX_regime"])


mixed_internal = internal_behavior_table(df_mixed)

print(mixed_internal.round(3))

In [ ]:
def category_mean_spread(df_in, regime):
    sub = df_in[df_in["HDX_regime"] == regime].copy()

    means = (
        sub.groupby(comfort_col)[x_col]
        .mean()
        .reindex(comfort_order)
        .dropna()
    )

    return means.max() - means.min(), means


spread_r1, means_r1 = category_mean_spread(df_mixed, "Regime 1")
spread_r2, means_r2 = category_mean_spread(df_mixed, "Regime 2")

print("Mixed walks only")
print("Regime 1 category mean spread:", spread_r1)
print("Regime 2 category mean spread:", spread_r2)
print("Spread ratio R1/R2:", spread_r1 / spread_r2)

print("\nRegime 1 means:")
print(means_r1.round(3))

print("\nRegime 2 means:")
print(means_r2.round(3))

In [ ]:
def kruskal_by_regime(df_in):
    rows = []

    for regime in ["Regime 1", "Regime 2"]:
        sub = df_in[df_in["HDX_regime"] == regime].copy()

        groups = []
        used_categories = []

        for cat in comfort_order:
            vals = sub.loc[sub[comfort_col] == cat, x_col].dropna().values

            if len(vals) >= 5:
                groups.append(vals)
                used_categories.append(cat)

        if len(groups) >= 2:
            H, p = stats.kruskal(*groups)
            N = sum(len(g) for g in groups)
            k = len(groups)
            epsilon2 = (H - k + 1) / (N - k)

            rows.append({
                "regime": regime,
                "N": N,
                "categories_used": len(used_categories),
                "H": H,
                "p": p,
                "epsilon2": epsilon2,
            })

    return pd.DataFrame(rows)


mixed_kw = kruskal_by_regime(df_mixed)

print(mixed_kw.round(5))

# test 2

In [ ]:
def spread_difference_stat(df_in, y_col):
    d = df_in.copy()

    def spread_for_regime(regime):
        sub = d[d["HDX_regime"] == regime].copy()

        means = (
            sub.groupby(y_col)[x_col]
            .mean()
            .reindex(comfort_order)
            .dropna()
        )

        if len(means) < 2:
            return np.nan

        return means.max() - means.min()

    s1 = spread_for_regime("Regime 1")
    s2 = spread_for_regime("Regime 2")

    return s1 - s2


# Use all data first
obs_stat = spread_difference_stat(df_test, comfort_col)

rng = np.random.default_rng(42)
n_perm = 2000
perm_stats = []

for _ in range(n_perm):
    d_perm = df_test.copy()

    # Shuffle comfort labels within each walk
    d_perm["comfort_perm"] = (
        d_perm.groupby("walk")[comfort_col]
        .transform(lambda s: rng.permutation(s.to_numpy()))
    )

    stat = spread_difference_stat(d_perm, "comfort_perm")
    perm_stats.append(stat)

perm_stats = np.array(perm_stats)
p_perm = np.mean(perm_stats >= obs_stat)

print("Observed spread difference R1 - R2:", obs_stat)
print("Permutation p-value:", p_perm)
print("Permutation 95% interval:", np.percentile(perm_stats, [2.5, 97.5]))

In [ ]:
obs_stat_mixed = spread_difference_stat(df_mixed, comfort_col)

rng = np.random.default_rng(42)
perm_stats_mixed = []

for _ in range(n_perm):
    d_perm = df_mixed.copy()

    d_perm["comfort_perm"] = (
        d_perm.groupby("walk")[comfort_col]
        .transform(lambda s: rng.permutation(s.to_numpy()))
    )

    stat = spread_difference_stat(d_perm, "comfort_perm")
    perm_stats_mixed.append(stat)

perm_stats_mixed = np.array(perm_stats_mixed)
p_perm_mixed = np.mean(perm_stats_mixed >= obs_stat_mixed)

print("Mixed walks observed spread difference R1 - R2:", obs_stat_mixed)
print("Mixed walks permutation p-value:", p_perm_mixed)
print("Mixed walks permutation 95% interval:", np.percentile(perm_stats_mixed, [2.5, 97.5]))

# Sembla que no hem trobat res però no passa re, crec k lo que ja hem trobat em serveix

In [ ]:
import statsmodels.formula.api as smf

d = df_test.copy()
d["TCV"] = d["thermal_comfort"].map(tcv_map)
d["high_regime"] = (d["HDX_regime"] == "Regime 2").astype(int)
d["HDX_centered"] = d[x_col] - d[x_col].mean()

model = smf.ols(
    "TCV ~ HDX_centered * high_regime + C(walk)",
    data=d
).fit(cov_type="cluster", cov_kwds={"groups": d["walk"]})

print(model.summary())

In [ ]:
d_mixed = df_mixed.copy()
d_mixed["TCV"] = d_mixed["thermal_comfort"].map(tcv_map)
d_mixed["high_regime"] = (d_mixed["HDX_regime"] == "Regime 2").astype(int)
d_mixed["HDX_centered"] = d_mixed[x_col] - d_mixed[x_col].mean()

model_mixed = smf.ols(
    "TCV ~ HDX_centered * high_regime + C(walk)",
    data=d_mixed
).fit(cov_type="cluster", cov_kwds={"groups": d_mixed["walk"]})

print(model_mixed.summary())

# Ara probarem per walk, en comptes de straight up HDX

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

x_col = "<HDX-HDX_fixed+<HDX>>"
comfort_col = "thermal_comfort"

comfort_order = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

boundary = 37.5

df_w = df_votes.copy()
df_w[x_col] = pd.to_numeric(df_w[x_col], errors="coerce")
df_w["walk"] = df_w["space_code"].astype(str).str[:2]
df_w["TCV"] = df_w[comfort_col].map(tcv_map)

df_w = df_w.dropna(subset=[x_col, "TCV", "walk"]).copy()

df_w["high_regime"] = (df_w[x_col] > boundary).astype(int)

def entropy_from_counts(values):
    counts = values.value_counts()
    p = counts / counts.sum()
    return -(p * np.log(p)).sum()

walk_summary = (
    df_w
    .groupby("walk")
    .agg(
        n=("TCV", "count"),
        mean_HDX=(x_col, "mean"),
        median_HDX=(x_col, "median"),
        std_HDX=(x_col, "std"),
        iqr_HDX=(x_col, lambda x: x.quantile(0.75) - x.quantile(0.25)),
        mean_TCV=("TCV", "mean"),
        std_TCV=("TCV", "std"),
        iqr_TCV=("TCV", lambda x: x.quantile(0.75) - x.quantile(0.25)),
        prop_high_regime=("high_regime", "mean"),
        n_categories=("TCV", "nunique"),
    )
    .reset_index()
)

entropy_table = (
    df_w
    .groupby("walk")[comfort_col]
    .apply(entropy_from_counts)
    .reset_index(name="comfort_entropy")
)

walk_summary = walk_summary.merge(entropy_table, on="walk")

walk_summary["walk_type"] = np.select(
    [
        walk_summary["prop_high_regime"] == 0,
        walk_summary["prop_high_regime"] == 1,
        (walk_summary["prop_high_regime"] > 0) & (walk_summary["prop_high_regime"] < 1),
    ],
    [
        "low-only",
        "high-only",
        "mixed",
    ],
    default="unknown",
)

print(walk_summary.round(3))

In [ ]:
for metric in ["std_TCV", "iqr_TCV", "comfort_entropy", "n_categories"]:
    print("\nMetric:", metric)

    groups = []
    labels = []

    for wt in ["low-only", "mixed", "high-only"]:
        vals = walk_summary.loc[walk_summary["walk_type"] == wt, metric].dropna()

        if len(vals) > 0:
            groups.append(vals)
            labels.append(wt)
            print(wt, "n walks =", len(vals), "mean =", vals.mean(), "median =", vals.median())

    if len(groups) >= 2:
        H, p = stats.kruskal(*groups)
        print("Kruskal-Wallis:", H, p)

In [ ]:
for metric in ["std_TCV", "iqr_TCV", "comfort_entropy", "n_categories"]:
    sub = walk_summary.dropna(subset=["mean_HDX", metric])

    rho, p = stats.spearmanr(sub["mean_HDX"], sub[metric])

    print(
        metric,
        "Spearman rho =", round(rho, 3),
        "p =", round(p, 4)
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for wt in ["low-only", "mixed", "high-only"]:
    g = walk_summary[walk_summary["walk_type"] == wt]

    plt.scatter(
        g["mean_HDX"],
        g["comfort_entropy"],
        s=g["n"] * 2,
        alpha=0.7,
        label=wt,
    )

plt.xlabel("Walk mean HDX")
plt.ylabel("Comfort entropy")
plt.title("Walk-level comfort heterogeneity vs mean HDX")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for wt in ["low-only", "mixed", "high-only"]:
    g = walk_summary[walk_summary["walk_type"] == wt]

    plt.scatter(
        g["mean_HDX"],
        g["std_TCV"],
        s=g["n"] * 2,
        alpha=0.7,
        label=wt,
    )

plt.xlabel("Walk mean HDX")
plt.ylabel("Within-walk comfort standard deviation")
plt.title("Comfort variability increases/decreases with walk-level HDX?")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# realment estem traient resultats? 

In [ ]:
import numpy as np
import pandas as pd

def normalized_category_separation(df_in, x_col, cat_col, regime_col, category_order=None):
    rows = []

    for regime, sub in df_in.groupby(regime_col):
        if category_order is None:
            cats = sub[cat_col].dropna().unique()
        else:
            cats = category_order

        means = (
            sub.groupby(cat_col)[x_col]
            .mean()
            .reindex(cats)
            .dropna()
        )

        regime_sd = sub[x_col].std()
        regime_iqr = sub[x_col].quantile(0.75) - sub[x_col].quantile(0.25)
        regime_range = sub[x_col].max() - sub[x_col].min()

        mean_spread = means.max() - means.min()

        rows.append({
            "regime": regime,
            "n": len(sub),
            "category_mean_spread": mean_spread,
            "regime_sd_HDX": regime_sd,
            "regime_iqr_HDX": regime_iqr,
            "regime_range_HDX": regime_range,
            "spread_over_sd": mean_spread / regime_sd if regime_sd > 0 else np.nan,
            "spread_over_iqr": mean_spread / regime_iqr if regime_iqr > 0 else np.nan,
            "spread_over_range": mean_spread / regime_range if regime_range > 0 else np.nan,
        })

    return pd.DataFrame(rows)

In [ ]:
norm_sep = normalized_category_separation(
    df_in=df_reg,  # or whatever dataframe has HDX_regime
    x_col="<HDX-HDX_fixed+<HDX>>",
    cat_col="thermal_comfort",
    regime_col="HDX_regime",
    category_order=[
        "Very comfortable",
        "Comfortable",
        "Slightly comfortable",
        "Neutral",
        "Slightly uncomfortable",
        "Uncomfortable",
        "Very uncomfortable",
    ],
)

print(norm_sep.round(3))

In [ ]:
from scipy import stats
import numpy as np

def regime_spearman_permutation(df_in, x_col, y_col, regime_col, n_perm=5000, random_state=0):
    rng = np.random.default_rng(random_state)
    rows = []

    for regime, sub in df_in.groupby(regime_col):
        sub = sub.dropna(subset=[x_col, y_col]).copy()

        obs_rho, obs_p = stats.spearmanr(sub[x_col], sub[y_col])

        perm_rhos = []

        y = sub[y_col].to_numpy()
        x = sub[x_col].to_numpy()

        for _ in range(n_perm):
            y_perm = rng.permutation(y)
            rho_perm, _ = stats.spearmanr(x, y_perm)
            perm_rhos.append(rho_perm)

        perm_rhos = np.array(perm_rhos)
        p_perm = np.mean(np.abs(perm_rhos) >= np.abs(obs_rho))

        rows.append({
            "regime": regime,
            "n": len(sub),
            "obs_spearman_rho": obs_rho,
            "ordinary_p": obs_p,
            "permutation_p": p_perm,
            "perm_2.5": np.percentile(perm_rhos, 2.5),
            "perm_97.5": np.percentile(perm_rhos, 97.5),
        })

    return pd.DataFrame(rows)

In [ ]:
tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

df_reg["TCV"] = df_reg["thermal_comfort"].map(tcv_map)

perm_regime = regime_spearman_permutation(
    df_in=df_reg,
    x_col="<HDX-HDX_fixed+<HDX>>",
    y_col="TCV",
    regime_col="HDX_regime",
)

print(perm_regime.round(4))

In [ ]:
def window_category_separation(df_in, x_col, cat_col, low, high, category_order):
    sub = df_in[(df_in[x_col] >= low) & (df_in[x_col] <= high)].copy()

    means = (
        sub.groupby(cat_col)[x_col]
        .mean()
        .reindex(category_order)
        .dropna()
    )

    if len(means) < 2:
        return None

    return {
        "low": low,
        "high": high,
        "width": high - low,
        "n": len(sub),
        "category_mean_spread": means.max() - means.min(),
        "hdx_sd": sub[x_col].std(),
        "hdx_iqr": sub[x_col].quantile(0.75) - sub[x_col].quantile(0.25),
        "spread_over_iqr": (means.max() - means.min()) / (
            sub[x_col].quantile(0.75) - sub[x_col].quantile(0.25)
        ),
    }

john = window_category_separation(
    df_in=df_reg,
    x_col="<HDX-HDX_fixed+<HDX>>",
    cat_col="thermal_comfort",
    low=38.5,
    high=42.0,
    category_order=[
        "Very comfortable",
        "Comfortable",
        "Slightly comfortable",
        "Neutral",
        "Slightly uncomfortable",
        "Uncomfortable",
        "Very uncomfortable",
    ]
)
print(john)
mike = window_category_separation(
    df_in=df_reg,
    x_col="<HDX-HDX_fixed+<HDX>>",
    cat_col="thermal_comfort",
    low=32.0,
    high=35.5,
    category_order=[
        "Very comfortable",
        "Comfortable",
        "Slightly comfortable",
        "Neutral",
        "Slightly uncomfortable",
        "Uncomfortable",
        "Very uncomfortable",
    ]
)
print(mike)
mike = window_category_separation(
    df_in=df_reg,
    x_col="<HDX-HDX_fixed+<HDX>>",
    cat_col="thermal_comfort",
    low=34.0,
    high=37.5,
    category_order=[
        "Very comfortable",
        "Comfortable",
        "Slightly comfortable",
        "Neutral",
        "Slightly uncomfortable",
        "Uncomfortable",
        "Very uncomfortable",
    ]
)
print(mike)

In [ ]:
from sklearn.metrics import roc_auc_score

def pair_metrics(df_in, x_col, cat_col, cat_a, cat_b, gender=None, gender_col="gender", bins=25):
    sub = df_in[df_in[cat_col].isin([cat_a, cat_b])].copy()
    if gender is not None:
        sub = sub[sub[gender_col] == gender].copy()

    x_a = sub.loc[sub[cat_col] == cat_a, x_col].dropna().to_numpy()
    x_b = sub.loc[sub[cat_col] == cat_b, x_col].dropna().to_numpy()

    if len(x_a) == 0 or len(x_b) == 0:
        return None

    y = np.r_[np.zeros(len(x_a)), np.ones(len(x_b))]
    x = np.r_[x_a, x_b]
    auc = roc_auc_score(y, x)
    overlap = histogram_overlap(x_a, x_b, bins=bins)

    return {
        "gender": gender if gender is not None else "All",
        "cat_a": cat_a,
        "cat_b": cat_b,
        "n_a": len(x_a),
        "n_b": len(x_b),
        "mean_a": x_a.mean(),
        "mean_b": x_b.mean(),
        "median_a": np.median(x_a),
        "median_b": np.median(x_b),
        "auc": auc,
        "overlap_hist": overlap,
        "delta_mean_b_minus_a": x_b.mean() - x_a.mean(),
        "delta_median_b_minus_a": np.median(x_b) - np.median(x_a),
    }

pairs_to_check = [
    (sensation_col, "Hot", "Very hot"),
    (comfort_col, "Comfortable", "Very comfortable"),
    (sensation_col, "Slightly cool", "Cool"),
]

rows = []
for cat_col, a, b in pairs_to_check:
    for g in [None, "Woman", "Man"]:
        out = pair_metrics(df, x_col, cat_col, a, b, gender=g)
        if out is not None:
            out["variable"] = cat_col
            rows.append(out)

pair_table = pd.DataFrame(rows)
print(pair_table)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# Tria la variable tèrmica
# x_col = "<T-T_fixed+<T>>"
x_col = "<HDX-HDX_fixed+<HDX>>"

comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"
gender_col = "gender"

sun_col = "sun"
wind_col = "wind"

sun_map = {
    "In full shade": -1,
    "In a mixture of sun and shadow": 0,
    "In full sun": 1,
}

wind_map = {
    "A strong wind": -1,
    "A moderate wind": -1,
    "A light wind": 0,
    "A very light wind": 0,
    "It's not windy": 1,
}

df = df_votes.copy()
df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
df["sun_num"] = df[sun_col].map(sun_map)
df["wind_num"] = df[wind_col].map(wind_map)

# definició simple de caminada
df["walk"] = df["space_code"].astype(str).str[:2]

df = df.dropna(subset=[x_col]).copy()


def ecdf_vals(x):
    x = np.sort(np.asarray(x))
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

def histogram_overlap(x1, x2, bins=25):
    all_x = np.concatenate([x1, x2])
    edges = np.histogram_bin_edges(all_x, bins=bins)
    h1, _ = np.histogram(x1, bins=edges, density=True)
    h2, _ = np.histogram(x2, bins=edges, density=True)
    widths = np.diff(edges)
    return np.sum(np.minimum(h1, h2) * widths)

def summarize_pair(x_a, x_b, cat_a, cat_b):
    return pd.DataFrame({
        "category": [cat_a, cat_b],
        "n": [len(x_a), len(x_b)],
        "mean": [x_a.mean(), x_b.mean()],
        "median": [np.median(x_a), np.median(x_b)],
        "std": [x_a.std(ddof=1) if len(x_a) > 1 else np.nan,
                x_b.std(ddof=1) if len(x_b) > 1 else np.nan],
        "q25": [np.quantile(x_a, 0.25), np.quantile(x_b, 0.25)],
        "q75": [np.quantile(x_a, 0.75), np.quantile(x_b, 0.75)],
    }).assign(iqr=lambda d: d["q75"] - d["q25"])

In [ ]:
def compare_pair_in_subset(
    df_in,
    x_col,
    cat_col,
    cat_a,
    cat_b,
    subset_mask=None,
    subset_name="All",
    bins=25,
    min_n=15,
):
    sub = df_in[df_in[cat_col].isin([cat_a, cat_b])].copy()

    if subset_mask is not None:
        sub = sub[subset_mask.loc[sub.index]].copy()

    x_a = sub.loc[sub[cat_col] == cat_a, x_col].dropna().to_numpy()
    x_b = sub.loc[sub[cat_col] == cat_b, x_col].dropna().to_numpy()

    print(f"\n=== {cat_a} vs {cat_b} | {subset_name} ===")

    if len(x_a) < min_n or len(x_b) < min_n:
        print(f"Too few data: {cat_a}={len(x_a)}, {cat_b}={len(x_b)}")
        return None

    summary = summarize_pair(x_a, x_b, cat_a, cat_b)
    print(summary)

    y = np.r_[np.zeros(len(x_a)), np.ones(len(x_b))]
    x = np.r_[x_a, x_b]
    auc = roc_auc_score(y, x)
    overlap = histogram_overlap(x_a, x_b, bins=bins)

    print(f"\nAUC ({cat_a}=0, {cat_b}=1): {auc:.3f}")
    print(f"Histogram overlap (~1 = very similar): {overlap:.3f}")
    print(f"Δmean ({cat_b} - {cat_a}): {x_b.mean() - x_a.mean():.3f}")
    print(f"Δmedian ({cat_b} - {cat_a}): {np.median(x_b) - np.median(x_a):.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].hist(x_a, bins=bins, alpha=0.5, density=True, label=f"{cat_a} (n={len(x_a)})")
    axes[0].hist(x_b, bins=bins, alpha=0.5, density=True, label=f"{cat_b} (n={len(x_b)})")
    axes[0].set_title(f"Histogram | {subset_name}")
    axes[0].set_xlabel(x_col)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    xa_s, ya = ecdf_vals(x_a)
    xb_s, yb = ecdf_vals(x_b)
    axes[1].plot(xa_s, ya, label=cat_a)
    axes[1].plot(xb_s, yb, label=cat_b)
    axes[1].set_title(f"ECDF | {subset_name}")
    axes[1].set_xlabel(x_col)
    axes[1].set_ylabel("Cumulative probability")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.suptitle(f"{cat_a} vs {cat_b} | {subset_name}")
    plt.tight_layout()
    plt.show()

    return {
        "subset": subset_name,
        "cat_a": cat_a,
        "cat_b": cat_b,
        "n_a": len(x_a),
        "n_b": len(x_b),
        "auc": auc,
        "overlap_hist": overlap,
        "delta_mean": x_b.mean() - x_a.mean(),
        "delta_median": np.median(x_b) - np.median(x_a),
    }

In [ ]:
def compare_pair_across_subgroups(
    df_in,
    x_col,
    cat_col,
    cat_a,
    cat_b,
    subgroup_col,
    subgroup_values,
    bins=25,
    min_n=15,
):
    rows = []
    for val in subgroup_values:
        mask = df_in[subgroup_col] == val
        out = compare_pair_in_subset(
            df_in=df_in,
            x_col=x_col,
            cat_col=cat_col,
            cat_a=cat_a,
            cat_b=cat_b,
            subset_mask=mask,
            subset_name=f"{subgroup_col} = {val}",
            bins=bins,
            min_n=min_n,
        )
        if out is not None:
            rows.append(out)

    if rows:
        res = pd.DataFrame(rows)
        print("\n=== Summary across subgroups ===")
        print(res)
        return res
    return None

In [ ]:
compare_two_categories(
    df_in=df,
    x_col=x_col,
    cat_col=comfort_col,
    cat_a="Comfortable",
    cat_b="Very comfortable",
    gender=None,
)

# Miro lo de distribuir-les/permutar random

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
temp_col = "<T-T_fixed+<T>>"
hdx_col = "<HDX-HDX_fixed+<HDX>>"

comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"

# Exemples típics; adapta'ls als teus mapes reals si cal
# IMPORTANT: fes servir els teus diccionaris reals si ja els tens definits
tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

tsv_map = {
    "Cool": 1,
    "Slightly cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}



def make_binned_null_band_plot(
    df_in: pd.DataFrame,
    x_col: str,
    y_col: str,
    y_map: dict,
    n_bins: int = 12,
    n_perm: int = 1000,
    min_count_per_bin: int = 8,
    title: str = "",
    x_label: str = "",
    y_label: str = "",
    random_state: int = 42,
):
    """
    Plot mean numeric vote vs x_col, with a null band from permutation.
    
    Null model:
      shuffle the votes across rows, keeping x fixed.
    """
    rng = np.random.default_rng(random_state)

    df = df_in.copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df["y_num"] = df[y_col].map(y_map)

    df = df.dropna(subset=[x_col, "y_num"]).copy()
    if df.empty:
        raise ValueError(f"No valid data for {x_col} and {y_col}")

    # Bin x
    # duplicates='drop' avoids crash if too many repeated values
    df["x_bin"] = pd.qcut(df[x_col], q=n_bins, duplicates="drop")

    # Bin centers
    bin_centers = (
        df.groupby("x_bin", observed=False)[x_col]
        .mean()
        .rename("x_center")
        .reset_index()
    )

    # Observed summary
    obs = (
        df.groupby("x_bin", observed=False)["y_num"]
        .agg(mean_y="mean", std_y="std", n="count")
        .reset_index()
    )
    obs = obs.merge(bin_centers, on="x_bin", how="left")
    obs["sem_y"] = obs["std_y"] / np.sqrt(obs["n"])
    obs = obs.sort_values("x_center").reset_index(drop=True)

    # Keep only bins with enough data
    keep_bins = obs.loc[obs["n"] >= min_count_per_bin, "x_bin"]
    df = df[df["x_bin"].isin(keep_bins)].copy()
    obs = obs[obs["x_bin"].isin(keep_bins)].copy().reset_index(drop=True)

    if obs.empty:
        raise ValueError("No bins left after min_count_per_bin filtering.")

    # Recompute in same order
    x_bins_order = obs["x_bin"].tolist()
    x_centers = obs["x_center"].to_numpy()
    y_obs = obs["mean_y"].to_numpy()
    y_sem = obs["sem_y"].to_numpy()

    # Permutation null curves
    perm_curves = []

    y_values = df["y_num"].to_numpy()

    for _ in range(n_perm):
        y_perm = rng.permutation(y_values)
        df_perm = df.copy()
        df_perm["y_perm"] = y_perm

        perm_summary = (
            df_perm.groupby("x_bin", observed=False)["y_perm"]
            .mean()
            .reindex(x_bins_order)
        )

        perm_curves.append(perm_summary.to_numpy())

    perm_curves = np.array(perm_curves)

    # Null band
    lower = np.nanpercentile(perm_curves, 2.5, axis=0)
    mid = np.nanpercentile(perm_curves, 50, axis=0)
    upper = np.nanpercentile(perm_curves, 97.5, axis=0)

    # Plot
    plt.figure(figsize=(8, 5))

    # null band
    plt.fill_between(x_centers, lower, upper, alpha=0.25, label="Permutation null 95% band")
    plt.plot(x_centers, mid, linestyle="--", alpha=0.8, label="Permutation median")

    # observed
    plt.errorbar(
        x_centers,
        y_obs,
        yerr=y_sem,
        fmt="o-",
        capsize=4,
        label="Observed mean ± SEM",
    )

    plt.xlabel(x_label if x_label else x_col)
    plt.ylabel(y_label if y_label else y_col)
    plt.title(title if title else f"{y_col} vs {x_col} with null band")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Return useful tables
    result = obs[["x_bin", "x_center", "mean_y", "std_y", "sem_y", "n"]].copy()
    result["null_p2.5"] = lower
    result["null_p50"] = mid
    result["null_p97.5"] = upper
    result["outside_null"] = (result["mean_y"] < result["null_p2.5"]) | (result["mean_y"] > result["null_p97.5"])

    return result, perm_curves

# TSV vs HDX

In [ ]:
res_sens_hdx, perm_sens_hdx = make_binned_null_band_plot(
    df_in=df_votes,
    x_col=hdx_col,
    y_col=sensation_col,
    y_map=tsv_map,
    n_bins=8,
    n_perm=1000,
    min_count_per_bin=8,
    title="Thermal sensation vs HDX with permutation null band",
    x_label="Corrected Humidex",
    y_label="Mean TSV",
    random_state=42,
)

print(res_sens_hdx)

# TSV vs T

In [ ]:
res_sens_t, perm_sens_t = make_binned_null_band_plot(
    df_in=df_votes,
    x_col=hdx_col,
    y_col=comfort_col,
    y_map=tcv_map,
    n_bins=10,
    n_perm=1000,
    min_count_per_bin=8,
    title="Thermal comfort vs HDX with permutation null band",
    x_label="Corrected Temperature",
    y_label="Mean TCV",
    random_state=42,
)

print(res_sens_t)

# TCV vs HDX

In [ ]:
res_comf_hdx, perm_comf_hdx = make_binned_null_band_plot(
    df_in=df_votes,
    x_col=hdx_col,
    y_col=comfort_col,
    y_map=tcv_map,
    n_bins=8,
    n_perm=1000,
    min_count_per_bin=8,
    title="Thermal comfort vs HDX with permutation null band",
    x_label="Corrected Humidex",
    y_label="Mean TCV",
    random_state=42,
)

print(res_comf_hdx)

In [ ]:
def make_binned_null_band_plot_on_ax(
    ax,
    df_in: pd.DataFrame,
    x_col: str,
    y_col: str,
    y_map: dict,
    n_bins: int = 12,
    n_perm: int = 1000,
    min_count_per_bin: int = 8,
    title: str = "",
    x_label: str = "",
    y_label: str = "",
    random_state: int = 42,
):
    rng = np.random.default_rng(random_state)

    df = df_in.copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df["y_num"] = df[y_col].map(y_map)
    df = df.dropna(subset=[x_col, "y_num"]).copy()

    df["x_bin"] = pd.qcut(df[x_col], q=n_bins, duplicates="drop")

    bin_centers = (
        df.groupby("x_bin", observed=False)[x_col]
        .mean()
        .rename("x_center")
        .reset_index()
    )

    obs = (
        df.groupby("x_bin", observed=False)["y_num"]
        .agg(mean_y="mean", std_y="std", n="count")
        .reset_index()
    )
    obs = obs.merge(bin_centers, on="x_bin", how="left")
    obs["sem_y"] = obs["std_y"] / np.sqrt(obs["n"])
    obs = obs.sort_values("x_center").reset_index(drop=True)

    keep_bins = obs.loc[obs["n"] >= min_count_per_bin, "x_bin"]
    df = df[df["x_bin"].isin(keep_bins)].copy()
    obs = obs[obs["x_bin"].isin(keep_bins)].copy().reset_index(drop=True)

    x_bins_order = obs["x_bin"].tolist()
    x_centers = obs["x_center"].to_numpy()
    y_obs = obs["mean_y"].to_numpy()
    y_sem = obs["sem_y"].to_numpy()

    perm_curves = []
    y_values = df["y_num"].to_numpy()

    for _ in range(n_perm):
        y_perm = rng.permutation(y_values)
        df_perm = df.copy()
        df_perm["y_perm"] = y_perm
        perm_summary = (
            df_perm.groupby("x_bin", observed=False)["y_perm"]
            .mean()
            .reindex(x_bins_order)
        )
        perm_curves.append(perm_summary.to_numpy())

    perm_curves = np.array(perm_curves)
    lower = np.nanpercentile(perm_curves, 2.5, axis=0)
    mid = np.nanpercentile(perm_curves, 50, axis=0)
    upper = np.nanpercentile(perm_curves, 97.5, axis=0)

    ax.fill_between(x_centers, lower, upper, alpha=0.25, label="Null 95% band")
    ax.plot(x_centers, mid, linestyle="--", alpha=0.8, label="Null median")
    ax.errorbar(
        x_centers,
        y_obs,
        yerr=y_sem,
        fmt="o-",
        capsize=4,
        label="Observed ± SEM",
    )
    ax.set_title(title)
    ax.set_xlabel(x_label if x_label else x_col)
    ax.set_ylabel(y_label if y_label else y_col)
    ax.grid(True, alpha=0.3)

    return obs


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

make_binned_null_band_plot_on_ax(
    axes[0, 0],
    df_votes, hdx_col, sensation_col, tsv_map,
    n_bins=10, n_perm=1000, min_count_per_bin=8,
    title="Thermal sensation vs HDX",
    x_label="Corrected Humidex",
    y_label="Mean TSV",
    random_state=42,
)

make_binned_null_band_plot_on_ax(
    axes[0, 1],
    df_votes, temp_col, sensation_col, tsv_map,
    n_bins=10, n_perm=1000, min_count_per_bin=8,
    title="Thermal sensation vs Temperature",
    x_label="Corrected Temperature",
    y_label="Mean TSV",
    random_state=42,
)

make_binned_null_band_plot_on_ax(
    axes[1, 0],
    df_votes, hdx_col, comfort_col, tcv_map,
    n_bins=10, n_perm=1000, min_count_per_bin=8,
    title="Thermal comfort vs HDX",
    x_label="Corrected Humidex",
    y_label="Mean TCV",
    random_state=42,
)

make_binned_null_band_plot_on_ax(
    axes[1, 1],
    df_votes, temp_col, comfort_col, tcv_map,
    n_bins=10, n_perm=1000, min_count_per_bin=8,
    title="Thermal comfort vs Temperature",
    x_label="Corrected Temperature",
    y_label="Mean TCV",
    random_state=42,
)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# TCV vs T

In [ ]:
res_comf_t, perm_comf_t = make_binned_null_band_plot(
    df_in=df_votes,
    x_col=temp_col,
    y_col=comfort_col,
    y_map=tcv_map,
    n_bins=10,
    n_perm=1000,
    min_count_per_bin=8,
    title="Thermal comfort vs Temperature with permutation null band",
    x_label="Corrected Temperature",
    y_label="Mean TCV",
    random_state=42,
)

print(res_comf_t)

# Ara amb l'altre plot

In [ ]:
def make_category_mean_x_null_plot(
    df_in: pd.DataFrame,
    x_col: str,
    y_col: str,
    y_map: dict,
    n_perm: int = 1000,
    min_count_per_cat: int = 8,
    title: str = "",
    x_label: str = "",
    y_label: str = "",
    random_state: int = 42,
):
    """
    Permute votes, keep x fixed, and plot:
      x-axis = mean x for each category
      y-axis = category order
    with a null band for the category mean x under permutation.
    """
    rng = np.random.default_rng(random_state)

    df = df_in.copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df["y_num"] = df[y_col].map(y_map)

    df = df.dropna(subset=[x_col, y_col, "y_num"]).copy()
    if df.empty:
        raise ValueError(f"No valid data for {x_col} and {y_col}")

    # observed summary
    obs = (
        df.groupby([y_col, "y_num"], as_index=False)[x_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("y_num")
    )

    obs = obs[obs["n"] >= min_count_per_cat].copy()
    obs["sem_x"] = obs["std_x"] / np.sqrt(obs["n"])

    if obs.empty:
        raise ValueError("No categories left after min_count_per_cat filtering.")

    cats_order = obs[y_col].tolist()
    y_positions = obs["y_num"].to_numpy()

    # restrict permutations to kept categories only
    df = df[df[y_col].isin(cats_order)].copy()

    perm_means = []

    y_values = df[y_col].to_numpy()

    for _ in range(n_perm):
        y_perm = rng.permutation(y_values)
        df_perm = df.copy()
        df_perm["y_perm"] = y_perm

        perm_summary = (
            df_perm.groupby("y_perm")[x_col]
            .mean()
            .reindex(cats_order)
        )
        perm_means.append(perm_summary.to_numpy())

    perm_means = np.array(perm_means)

    lower = np.nanpercentile(perm_means, 2.5, axis=0)
    mid = np.nanpercentile(perm_means, 50, axis=0)
    upper = np.nanpercentile(perm_means, 97.5, axis=0)

    # plot
    plt.figure(figsize=(8, 5))

    # null band is horizontal along y categories, so fill_betweenx
    plt.fill_betweenx(
        y_positions,
        lower,
        upper,
        alpha=0.25,
        label="Permutation null 95% band",
    )
    plt.plot(mid, y_positions, linestyle="--", alpha=0.8, label="Permutation median")

    plt.errorbar(
        obs["mean_x"],
        y_positions,
        xerr=obs["sem_x"],
        fmt="o-",
        capsize=4,
        label="Observed mean ± SEM",
    )

    plt.yticks(y_positions, cats_order)
    plt.xlabel(x_label if x_label else x_col)
    plt.ylabel(y_label if y_label else y_col)
    plt.title(title if title else f"Mean {x_col} by {y_col} with null band")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    result = obs[[y_col, "y_num", "mean_x", "std_x", "sem_x", "n"]].copy()
    result["null_p2.5"] = lower
    result["null_p50"] = mid
    result["null_p97.5"] = upper
    result["outside_null"] = (result["mean_x"] < result["null_p2.5"]) | (result["mean_x"] > result["null_p97.5"])

    return result, perm_means

In [ ]:
def make_category_mean_x_null_plot_on_ax(
    ax,
    df_in: pd.DataFrame,
    x_col: str,
    y_col: str,
    y_map: dict,
    n_perm: int = 1000,
    min_count_per_cat: int = 8,
    title: str = "",
    x_label: str = "",
    y_label: str = "",
    random_state: int = 42,
):
    rng = np.random.default_rng(random_state)

    df = df_in.copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df["y_num"] = df[y_col].map(y_map)
    df = df.dropna(subset=[x_col, y_col, "y_num"]).copy()

    obs = (
        df.groupby([y_col, "y_num"], as_index=False)[x_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("y_num")
    )

    obs = obs[obs["n"] >= min_count_per_cat].copy()
    obs["sem_x"] = obs["std_x"] / np.sqrt(obs["n"])

    cats_order = obs[y_col].tolist()
    y_positions = obs["y_num"].to_numpy()

    df = df[df[y_col].isin(cats_order)].copy()

    perm_means = []
    y_values = df[y_col].to_numpy()

    for _ in range(n_perm):
        y_perm = rng.permutation(y_values)
        df_perm = df.copy()
        df_perm["y_perm"] = y_perm
        perm_summary = (
            df_perm.groupby("y_perm")[x_col]
            .mean()
            .reindex(cats_order)
        )
        perm_means.append(perm_summary.to_numpy())

    perm_means = np.array(perm_means)

    lower = np.nanpercentile(perm_means, 2.5, axis=0)
    mid = np.nanpercentile(perm_means, 50, axis=0)
    upper = np.nanpercentile(perm_means, 97.5, axis=0)

    ax.fill_betweenx(y_positions, lower, upper, alpha=0.25, label="Null 95% band")
    ax.plot(mid, y_positions, linestyle="--", alpha=0.8, label="Null median")
    ax.errorbar(
        obs["mean_x"],
        y_positions,
        xerr=obs["sem_x"],
        fmt="o-",
        capsize=4,
        label="Observed ± SEM",
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(cats_order)
    ax.set_xlabel(x_label if x_label else x_col)
    ax.set_ylabel(y_label if y_label else y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    return obs


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

make_category_mean_x_null_plot_on_ax(
    axes[0, 0],
    df_votes, hdx_col, sensation_col, tsv_map,
    n_perm=1000, min_count_per_cat=8,
    title="Mean HDX by thermal sensation",
    x_label="Corrected Humidex",
    y_label="Thermal sensation",
    random_state=42,
)

make_category_mean_x_null_plot_on_ax(
    axes[0, 1],
    df_votes, temp_col, sensation_col, tsv_map,
    n_perm=1000, min_count_per_cat=8,
    title="Mean Temperature by thermal sensation",
    x_label="Corrected Temperature",
    y_label="Thermal sensation",
    random_state=42,
)

make_category_mean_x_null_plot_on_ax(
    axes[1, 0],
    df_votes, hdx_col, comfort_col, tcv_map,
    n_perm=1000, min_count_per_cat=8,
    title="Mean HDX by thermal comfort",
    x_label="Corrected Humidex",
    y_label="Thermal comfort",
    random_state=42,
)

make_category_mean_x_null_plot_on_ax(
    axes[1, 1],
    df_votes, temp_col, comfort_col, tcv_map,
    n_perm=1000, min_count_per_cat=8,
    title="Mean Temperature by thermal comfort",
    x_label="Corrected Temperature",
    y_label="Thermal comfort",
    random_state=42,
)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
temp_abs_col = "T_K"   # absolute temperature column for walk-relative temperature
hdx_col = "<HDX-HDX_fixed+<HDX>>"   # corrected HDX, used directly

comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"

# Use your real maps if already defined
tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

tsv_map = {
    "Cool": 1,
    "Slightly cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}

df_votes["T_K"] = df_votes["<T-T_fixed+<T>>"] + 273.15  # example conversion to Kelvin if temp_col is in Celsius

# =========================
# HELPERS
# =========================
def prepare_temperature_relative_by_walk(df_in, temp_abs_col, walk_col="walk"):
    df = df_in.copy()
    df[temp_abs_col] = pd.to_numeric(df[temp_abs_col], errors="coerce")
    df = df.dropna(subset=[temp_abs_col, walk_col]).copy()

    walk_mean = df.groupby(walk_col)[temp_abs_col].transform("mean")
    rel_col = f"{temp_abs_col}__rel_walk"
    df[rel_col] = df[temp_abs_col] - walk_mean
    return df, rel_col


def make_category_mean_x_null_plot_on_ax(
    ax,
    df_in: pd.DataFrame,
    x_col: str,
    y_col: str,
    y_map: dict,
    n_perm: int = 1000,
    min_count_per_cat: int = 8,
    title: str = "",
    x_label: str = "",
    y_label: str = "",
    random_state: int = 42,
):
    """
    Plot:
      y-axis = categories
      x-axis = mean x for each category
    Null model:
      shuffle the votes across rows, keeping x fixed.
    """
    rng = np.random.default_rng(random_state)

    df = df_in.copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df["y_num"] = df[y_col].map(y_map)
    df = df.dropna(subset=[x_col, y_col, "y_num"]).copy()

    if df.empty:
        ax.set_title(title + " (no data)")
        return None

    # Observed summary
    obs = (
        df.groupby([y_col, "y_num"], as_index=False)[x_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("y_num")
    )

    obs = obs[obs["n"] >= min_count_per_cat].copy()
    if obs.empty:
        ax.set_title(title + " (no categories after filtering)")
        return None

    obs["sem_x"] = obs["std_x"] / np.sqrt(obs["n"])

    cats_order = obs[y_col].tolist()
    y_positions = obs["y_num"].to_numpy()

    # Restrict permutations to kept categories only
    df = df[df[y_col].isin(cats_order)].copy()

    perm_means = []
    y_values = df[y_col].to_numpy()

    for _ in range(n_perm):
        y_perm = rng.permutation(y_values)
        df_perm = df.copy()
        df_perm["y_perm"] = y_perm

        perm_summary = (
            df_perm.groupby("y_perm")[x_col]
            .mean()
            .reindex(cats_order)
        )
        perm_means.append(perm_summary.to_numpy())

    perm_means = np.array(perm_means)

    lower = np.nanpercentile(perm_means, 2.5, axis=0)
    mid = np.nanpercentile(perm_means, 50, axis=0)
    upper = np.nanpercentile(perm_means, 97.5, axis=0)

    # Plot
    ax.fill_betweenx(
        y_positions, lower, upper,
        alpha=0.25, label="Null 95% band"
    )
    ax.plot(mid, y_positions, linestyle="--", alpha=0.8, label="Null median")
    ax.errorbar(
        obs["mean_x"],
        y_positions,
        xerr=obs["sem_x"],
        fmt="o-",
        capsize=4,
        label="Observed ± SEM",
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(cats_order)
    ax.set_xlabel(x_label if x_label else x_col)
    ax.set_ylabel(y_label if y_label else y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    result = obs[[y_col, "y_num", "mean_x", "std_x", "sem_x", "n"]].copy()
    result["null_p2.5"] = lower
    result["null_p50"] = mid
    result["null_p97.5"] = upper
    result["outside_null"] = (
        (result["mean_x"] < result["null_p2.5"]) |
        (result["mean_x"] > result["null_p97.5"])
    )

    return result


# =========================
# PREPARE DATA
# =========================
df_plot = df_votes.copy()
df_plot["walk"] = df_plot["space_code"].astype(str).str[:2]

# temperature relative to walk mean
df_plot, temp_rel_walk_col = prepare_temperature_relative_by_walk(
    df_plot,
    temp_abs_col=temp_abs_col,
    walk_col="walk"
)

# HDX used directly
df_plot[hdx_col] = pd.to_numeric(df_plot[hdx_col], errors="coerce")


# =========================
# 2x2 PLOT
# =========================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

res_1 = make_category_mean_x_null_plot_on_ax(
    axes[0, 0],
    df_plot,
    hdx_col,
    sensation_col,
    tsv_map,
    n_perm=1000,
    min_count_per_cat=8,
    title="Mean HDX by thermal sensation",
    x_label="Corrected Humidex",
    y_label="Thermal sensation",
    random_state=42,
)

res_2 = make_category_mean_x_null_plot_on_ax(
    axes[0, 1],
    df_plot,
    temp_rel_walk_col,
    sensation_col,
    tsv_map,
    n_perm=1000,
    min_count_per_cat=8,
    title="Mean temperature relative to walk mean by thermal sensation",
    x_label="Temperature relative to walk mean",
    y_label="Thermal sensation",
    random_state=42,
)

res_3 = make_category_mean_x_null_plot_on_ax(
    axes[1, 0],
    df_plot,
    hdx_col,
    comfort_col,
    tcv_map,
    n_perm=1000,
    min_count_per_cat=8,
    title="Mean HDX by thermal comfort",
    x_label="Corrected Humidex",
    y_label="Thermal comfort",
    random_state=42,
)

res_4 = make_category_mean_x_null_plot_on_ax(
    axes[1, 1],
    df_plot,
    temp_rel_walk_col,
    comfort_col,
    tcv_map,
    n_perm=1000,
    min_count_per_cat=8,
    title="Mean temperature relative to walk mean by thermal comfort",
    x_label="Temperature relative to walk mean",
    y_label="Thermal comfort",
    random_state=42,
)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


# Optional: inspect tables
print("\n=== Sensation vs HDX ===")
print(res_1)

print("\n=== Sensation vs walk-relative Temperature ===")
print(res_2)

print("\n=== Comfort vs HDX ===")
print(res_3)

print("\n=== Comfort vs walk-relative Temperature ===")
print(res_4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
temp_abs_col = "T_K"   # absolute temperature column for walk-relative temperature
hdx_col = "<HDX-HDX_fixed+<HDX>>"   # corrected HDX, used directly

comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"

# Original maps
tcv_map_grouped = {
    "Comfortable / Very comfortable": 1,
    "Slightly comfortable": 2,
    "Neutral": 3,
    "Slightly uncomfortable": 4,
    "Uncomfortable": 5,
    "Very uncomfortable": 6,
}

tsv_map_grouped = {
    "Cool / Slightly cool": 1,
    "Neutral": 2,
    "Slightly warm": 3,
    "Warm": 4,
    "Hot": 5,
    "Very hot": 6,
}

# Grouping rules
comfort_group_map = {
    "Very comfortable": "Comfortable / Very comfortable",
    "Comfortable": "Comfortable / Very comfortable",
    "Slightly comfortable": "Slightly comfortable",
    "Neutral": "Neutral",
    "Slightly uncomfortable": "Slightly uncomfortable",
    "Uncomfortable": "Uncomfortable",
    "Very uncomfortable": "Very uncomfortable",
}

sensation_group_map = {
    "Cool": "Cool / Slightly cool",
    "Slightly cool": "Cool / Slightly cool",
    "Neutral": "Neutral",
    "Slightly warm": "Slightly warm",
    "Warm": "Warm",
    "Hot": "Hot",
    "Very hot": "Very hot",
}


# =========================
# HELPERS
# =========================
def prepare_temperature_relative_by_walk(df_in, temp_abs_col, walk_col="walk"):
    df = df_in.copy()
    df[temp_abs_col] = pd.to_numeric(df[temp_abs_col], errors="coerce")
    df = df.dropna(subset=[temp_abs_col, walk_col]).copy()

    walk_mean = df.groupby(walk_col)[temp_abs_col].transform("mean")
    rel_col = f"{temp_abs_col}__rel_walk"
    df[rel_col] = df[temp_abs_col] - walk_mean
    return df, rel_col


def make_category_mean_x_null_plot_on_ax(
    ax,
    df_in: pd.DataFrame,
    x_col: str,
    y_col: str,
    y_map: dict,
    n_perm: int = 1000,
    min_count_per_cat: int = 8,
    title: str = "",
    x_label: str = "",
    y_label: str = "",
    random_state: int = 42,
):
    rng = np.random.default_rng(random_state)

    df = df_in.copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df["y_num"] = df[y_col].map(y_map)
    df = df.dropna(subset=[x_col, y_col, "y_num"]).copy()

    if df.empty:
        ax.set_title(title + " (no data)")
        return None

    obs = (
        df.groupby([y_col, "y_num"], as_index=False)[x_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("y_num")
    )

    obs = obs[obs["n"] >= min_count_per_cat].copy()
    if obs.empty:
        ax.set_title(title + " (no categories after filtering)")
        return None

    obs["sem_x"] = obs["std_x"] / np.sqrt(obs["n"])

    cats_order = obs[y_col].tolist()
    y_positions = obs["y_num"].to_numpy()

    # Restrict permutations to kept categories only
    df = df[df[y_col].isin(cats_order)].copy()

    perm_means = []
    y_values = df[y_col].to_numpy()

    for _ in range(n_perm):
        y_perm = rng.permutation(y_values)
        df_perm = df.copy()
        df_perm["y_perm"] = y_perm

        perm_summary = (
            df_perm.groupby("y_perm")[x_col]
            .mean()
            .reindex(cats_order)
        )
        perm_means.append(perm_summary.to_numpy())

    perm_means = np.array(perm_means)

    lower = np.nanpercentile(perm_means, 2.5, axis=0)
    mid = np.nanpercentile(perm_means, 50, axis=0)
    upper = np.nanpercentile(perm_means, 97.5, axis=0)

    ax.fill_betweenx(
        y_positions, lower, upper,
        alpha=0.25, label="Null 95% band"
    )
    ax.plot(mid, y_positions, linestyle="--", alpha=0.8, label="Null median")
    ax.errorbar(
        obs["mean_x"],
        y_positions,
        xerr=obs["sem_x"],
        fmt="o-",
        capsize=4,
        label="Observed ± SEM",
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(cats_order)
    ax.set_xlabel(x_label if x_label else x_col)
    ax.set_ylabel(y_label if y_label else y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    result = obs[[y_col, "y_num", "mean_x", "std_x", "sem_x", "n"]].copy()
    result["null_p2.5"] = lower
    result["null_p50"] = mid
    result["null_p97.5"] = upper
    result["outside_null"] = (
        (result["mean_x"] < result["null_p2.5"]) |
        (result["mean_x"] > result["null_p97.5"])
    )

    return result


# =========================
# PREPARE DATA
# =========================
df_plot = df_votes.copy()
df_plot["walk"] = df_plot["space_code"].astype(str).str[:2]

# Group categories
df_plot["thermal_comfort_grouped"] = df_plot[comfort_col].map(comfort_group_map)
df_plot["thermal_sensation_grouped"] = df_plot[sensation_col].map(sensation_group_map)

# Temperature relative to walk mean
df_plot, temp_rel_walk_col = prepare_temperature_relative_by_walk(
    df_plot,
    temp_abs_col=temp_abs_col,
    walk_col="walk"
)

# HDX used directly
df_plot[hdx_col] = pd.to_numeric(df_plot[hdx_col], errors="coerce")


# =========================
# 2x2 PLOT
# =========================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

res_1 = make_category_mean_x_null_plot_on_ax(
    axes[0, 0],
    df_plot,
    hdx_col,
    "thermal_sensation_grouped",
    tsv_map_grouped,
    n_perm=1000,
    min_count_per_cat=8,
    title="Mean HDX by grouped thermal sensation",
    x_label="Corrected Humidex",
    y_label="Grouped thermal sensation",
    random_state=42,
)

res_2 = make_category_mean_x_null_plot_on_ax(
    axes[0, 1],
    df_plot,
    temp_rel_walk_col,
    "thermal_sensation_grouped",
    tsv_map_grouped,
    n_perm=1000,
    min_count_per_cat=8,
    title="Mean temperature relative to walk mean by grouped thermal sensation",
    x_label="Temperature relative to walk mean",
    y_label="Grouped thermal sensation",
    random_state=42,
)

res_3 = make_category_mean_x_null_plot_on_ax(
    axes[1, 0],
    df_plot,
    hdx_col,
    "thermal_comfort_grouped",
    tcv_map_grouped,
    n_perm=1000,
    min_count_per_cat=8,
    title="Mean HDX by grouped thermal comfort",
    x_label="Corrected Humidex",
    y_label="Grouped thermal comfort",
    random_state=42,
)

res_4 = make_category_mean_x_null_plot_on_ax(
    axes[1, 1],
    df_plot,
    temp_rel_walk_col,
    "thermal_comfort_grouped",
    tcv_map_grouped,
    n_perm=1000,
    min_count_per_cat=8,
    title="Mean temperature relative to walk mean by grouped thermal comfort",
    x_label="Temperature relative to walk mean",
    y_label="Grouped thermal comfort",
    random_state=42,
)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


# Optional: inspect tables
print("\n=== Grouped sensation vs HDX ===")
print(res_1)

print("\n=== Grouped sensation vs walk-relative Temperature ===")
print(res_2)

print("\n=== Grouped comfort vs HDX ===")
print(res_3)

print("\n=== Grouped comfort vs walk-relative Temperature ===")
print(res_4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
temp_abs_col = "T_K"   # absolute temperature column for walk-relative temperature
hdx_col = "<HDX-HDX_fixed+<HDX>>"   # corrected HDX, used directly

comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"
gender_col = "gender"

valid_genders = ["Woman", "Man"]

# Grouped maps
tcv_map_grouped = {
    "Comfortable / Very comfortable": 1,
    "Slightly comfortable": 2,
    "Neutral": 3,
    "Slightly uncomfortable": 4,
    "Uncomfortable": 5,
    "Very uncomfortable": 6,
}

tsv_map_grouped = {
    "Cool / Slightly cool": 1,
    "Neutral": 2,
    "Slightly warm": 3,
    "Warm": 4,
    "Hot": 5,
    "Very hot": 6,
}

# Grouping rules
comfort_group_map = {
    "Very comfortable": "Comfortable / Very comfortable",
    "Comfortable": "Comfortable / Very comfortable",
    "Slightly comfortable": "Slightly comfortable",
    "Neutral": "Neutral",
    "Slightly uncomfortable": "Slightly uncomfortable",
    "Uncomfortable": "Uncomfortable",
    "Very uncomfortable": "Very uncomfortable",
}

sensation_group_map = {
    "Cool": "Cool / Slightly cool",
    "Slightly cool": "Cool / Slightly cool",
    "Neutral": "Neutral",
    "Slightly warm": "Slightly warm",
    "Warm": "Warm",
    "Hot": "Hot",
    "Very hot": "Very hot",
}


# =========================
# HELPERS
# =========================
def prepare_temperature_relative_by_walk(df_in, temp_abs_col, walk_col="walk"):
    df = df_in.copy()
    df[temp_abs_col] = pd.to_numeric(df[temp_abs_col], errors="coerce")
    df = df.dropna(subset=[temp_abs_col, walk_col]).copy()

    walk_mean = df.groupby(walk_col)[temp_abs_col].transform("mean")
    rel_col = f"{temp_abs_col}__rel_walk"
    df[rel_col] = df[temp_abs_col] - walk_mean
    return df, rel_col


def make_category_mean_x_null_plot_on_ax(
    ax,
    df_in: pd.DataFrame,
    x_col: str,
    y_col: str,
    y_map: dict,
    n_perm: int = 1000,
    min_count_per_cat: int = 8,
    title: str = "",
    x_label: str = "",
    y_label: str = "",
    random_state: int = 42,
):
    rng = np.random.default_rng(random_state)

    df = df_in.copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df["y_num"] = df[y_col].map(y_map)
    df = df.dropna(subset=[x_col, y_col, "y_num"]).copy()

    if df.empty:
        ax.set_title(title + " (no data)")
        return None

    obs = (
        df.groupby([y_col, "y_num"], as_index=False)[x_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("y_num")
    )

    obs = obs[obs["n"] >= min_count_per_cat].copy()
    if obs.empty:
        ax.set_title(title + " (no categories after filtering)")
        return None

    obs["sem_x"] = obs["std_x"] / np.sqrt(obs["n"])

    cats_order = obs[y_col].tolist()
    y_positions = obs["y_num"].to_numpy()

    # Restrict permutations to kept categories only
    df = df[df[y_col].isin(cats_order)].copy()

    perm_means = []
    y_values = df[y_col].to_numpy()

    for _ in range(n_perm):
        y_perm = rng.permutation(y_values)
        df_perm = df.copy()
        df_perm["y_perm"] = y_perm

        perm_summary = (
            df_perm.groupby("y_perm")[x_col]
            .mean()
            .reindex(cats_order)
        )
        perm_means.append(perm_summary.to_numpy())

    perm_means = np.array(perm_means)

    lower = np.nanpercentile(perm_means, 2.5, axis=0)
    mid = np.nanpercentile(perm_means, 50, axis=0)
    upper = np.nanpercentile(perm_means, 97.5, axis=0)

    ax.fill_betweenx(
        y_positions, lower, upper,
        alpha=0.25, label="Null 95% band"
    )
    ax.plot(mid, y_positions, linestyle="--", alpha=0.8, label="Null median")
    ax.errorbar(
        obs["mean_x"],
        y_positions,
        xerr=obs["sem_x"],
        fmt="o-",
        capsize=4,
        label="Observed ± SEM",
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(cats_order)
    ax.set_xlabel(x_label if x_label else x_col)
    ax.set_ylabel(y_label if y_label else y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    result = obs[[y_col, "y_num", "mean_x", "std_x", "sem_x", "n"]].copy()
    result["null_p2.5"] = lower
    result["null_p50"] = mid
    result["null_p97.5"] = upper
    result["outside_null"] = (
        (result["mean_x"] < result["null_p2.5"]) |
        (result["mean_x"] > result["null_p97.5"])
    )

    return result


def plot_grouped_gender_2x2(
    df_in,
    gender_value,
    temp_abs_col="T_K",
    hdx_col="<HDX-HDX_fixed+<HDX>>",
    comfort_col="thermal_comfort",
    sensation_col="thermal_sensation",
    gender_col="gender",
    n_perm=1000,
    min_count_per_cat=8,
    random_state=42,
):
    sub = df_in[df_in[gender_col] == gender_value].copy()
    if sub.empty:
        print(f"No data for gender={gender_value}")
        return None

    sub["walk"] = sub["space_code"].astype(str).str[:2]

    # grouped categories
    sub["thermal_comfort_grouped"] = sub[comfort_col].map(comfort_group_map)
    sub["thermal_sensation_grouped"] = sub[sensation_col].map(sensation_group_map)

    # walk-relative temperature
    sub, temp_rel_walk_col = prepare_temperature_relative_by_walk(
        sub,
        temp_abs_col=temp_abs_col,
        walk_col="walk"
    )

    # HDX direct
    sub[hdx_col] = pd.to_numeric(sub[hdx_col], errors="coerce")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    res_1 = make_category_mean_x_null_plot_on_ax(
        axes[0, 0],
        sub,
        hdx_col,
        "thermal_sensation_grouped",
        tsv_map_grouped,
        n_perm=n_perm,
        min_count_per_cat=min_count_per_cat,
        title=f"{gender_value}: Mean HDX by grouped thermal sensation",
        x_label="Corrected Humidex",
        y_label="Grouped thermal sensation",
        random_state=random_state,
    )

    res_2 = make_category_mean_x_null_plot_on_ax(
        axes[0, 1],
        sub,
        temp_rel_walk_col,
        "thermal_sensation_grouped",
        tsv_map_grouped,
        n_perm=n_perm,
        min_count_per_cat=min_count_per_cat,
        title=f"{gender_value}: Mean temperature relative to walk mean by grouped thermal sensation",
        x_label="Temperature relative to walk mean",
        y_label="Grouped thermal sensation",
        random_state=random_state,
    )

    res_3 = make_category_mean_x_null_plot_on_ax(
        axes[1, 0],
        sub,
        hdx_col,
        "thermal_comfort_grouped",
        tcv_map_grouped,
        n_perm=n_perm,
        min_count_per_cat=min_count_per_cat,
        title=f"{gender_value}: Mean HDX by grouped thermal comfort",
        x_label="Corrected Humidex",
        y_label="Grouped thermal comfort",
        random_state=random_state,
    )

    res_4 = make_category_mean_x_null_plot_on_ax(
        axes[1, 1],
        sub,
        temp_rel_walk_col,
        "thermal_comfort_grouped",
        tcv_map_grouped,
        n_perm=n_perm,
        min_count_per_cat=min_count_per_cat,
        title=f"{gender_value}: Mean temperature relative to walk mean by grouped thermal comfort",
        x_label="Temperature relative to walk mean",
        y_label="Grouped thermal comfort",
        random_state=random_state,
    )

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    return {
        "sensation_hdx": res_1,
        "sensation_temp_rel_walk": res_2,
        "comfort_hdx": res_3,
        "comfort_temp_rel_walk": res_4,
    }


# =========================
# PREPARE INPUT DATA
# =========================
df_gender = df_votes.copy()
df_gender = df_gender[df_gender[gender_col].isin(valid_genders)].copy()

# =========================
# RUN FOR WOMAN
# =========================
results_woman = plot_grouped_gender_2x2(
    df_in=df_gender,
    gender_value="Woman",
    temp_abs_col=temp_abs_col,
    hdx_col=hdx_col,
    comfort_col=comfort_col,
    sensation_col=sensation_col,
    gender_col=gender_col,
    n_perm=1000,
    min_count_per_cat=8,
    random_state=42,
)

# Optional tables
print("\n=== WOMAN | grouped sensation vs HDX ===")
print(results_woman["sensation_hdx"])

print("\n=== WOMAN | grouped sensation vs walk-relative Temperature ===")
print(results_woman["sensation_temp_rel_walk"])

print("\n=== WOMAN | grouped comfort vs HDX ===")
print(results_woman["comfort_hdx"])

print("\n=== WOMAN | grouped comfort vs walk-relative Temperature ===")
print(results_woman["comfort_temp_rel_walk"])


# =========================
# RUN FOR MAN
# =========================
results_man = plot_grouped_gender_2x2(
    df_in=df_gender,
    gender_value="Man",
    temp_abs_col=temp_abs_col,
    hdx_col=hdx_col,
    comfort_col=comfort_col,
    sensation_col=sensation_col,
    gender_col=gender_col,
    n_perm=1000,
    min_count_per_cat=8,
    random_state=42,
)

# Optional tables
print("\n=== MAN | grouped sensation vs HDX ===")
print(results_man["sensation_hdx"])

print("\n=== MAN | grouped sensation vs walk-relative Temperature ===")
print(results_man["sensation_temp_rel_walk"])

print("\n=== MAN | grouped comfort vs HDX ===")
print(results_man["comfort_hdx"])

print("\n=== MAN | grouped comfort vs walk-relative Temperature ===")
print(results_man["comfort_temp_rel_walk"])

# Probem otro

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# =========================
# CONFIG
# =========================
temp_abs_col = "T_K"                  # per fer T_rel_walk
temp_corr_col = "<T-T_fixed+<T>>"     # si també vols provar la corregida/global
hdx_col = "<HDX-HDX_fixed+<HDX>>"

comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"

# =========================
# PREPARE DATA
# =========================
df = df_votes.copy()
df["walk"] = df["space_code"].astype(str).str[:2]

for c in [temp_abs_col, temp_corr_col, hdx_col]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Temperature relative to walk mean
df["T_rel_walk"] = df[temp_abs_col] - df.groupby("walk")[temp_abs_col].transform("mean")

# 3-group comfort
def comfort_side(cat):
    if cat in ["Very comfortable", "Comfortable", "Slightly comfortable"]:
        return "comfortable_side"
    elif cat == "Neutral":
        return "neutral"
    elif cat in ["Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"]:
        return "uncomfortable_side"
    return np.nan

df["comfort_3"] = df[comfort_col].map(comfort_side)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from scipy import stats


# ============================================================
# CONFIG
# ============================================================

temp_abs_col = "T_K"
temp_corr_col = "<T-T_fixed+<T>>"
hdx_col = "<HDX-HDX_fixed+<HDX>>"

comfort_col = "thermal_comfort"

class_order = [
    "comfortable_side",
    "neutral",
    "uncomfortable_side",
]


# ============================================================
# PREPARE DATA
# ============================================================

df = df_votes.copy()
df["walk"] = df["space_code"].astype(str).str[:2]

for c in [temp_abs_col, temp_corr_col, hdx_col]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["T_rel_walk"] = (
    df[temp_abs_col]
    - df.groupby("walk")[temp_abs_col].transform("mean")
)


def comfort_side(cat):
    if cat in ["Very comfortable", "Comfortable", "Slightly comfortable"]:
        return "comfortable_side"
    elif cat == "Neutral":
        return "neutral"
    elif cat in ["Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"]:
        return "uncomfortable_side"
    return np.nan


df["comfort_3"] = df[comfort_col].map(comfort_side)


# ============================================================
# HELPERS
# ============================================================

def wilson_ci(k, n, confidence=0.95):
    if n == 0:
        return np.nan, np.nan

    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    phat = k / n

    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (
        z
        * np.sqrt((phat * (1 - phat) / n) + (z**2 / (4 * n**2)))
        / denom
    )

    return center - half, center + half


def predict_proba_fixed_order(model, x_grid, class_order):
    probs = model.predict_proba(x_grid.reshape(-1, 1))

    prob_df = pd.DataFrame(probs, columns=model.classes_)

    for c in class_order:
        if c not in prob_df.columns:
            prob_df[c] = np.nan

    return prob_df[class_order].to_numpy()


def crossing_x(x, y):
    """
    Estimate x where y crosses 0.
    If no crossing exists, returns the x where |y| is smallest.
    """
    y = np.asarray(y)
    x = np.asarray(x)

    sign_change = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]

    if len(sign_change) == 0:
        return x[np.nanargmin(np.abs(y))]

    i = sign_change[0]

    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]

    if y1 == y0:
        return x0

    return x0 - y0 * (x1 - x0) / (y1 - y0)


# ============================================================
# MAIN FUNCTION WITH WALK-LEVEL BOOTSTRAP
# ============================================================

def fit_comfort_side_model_with_error(
    df_in,
    x_col,
    y_col="comfort_3",
    cluster_col="walk",
    n_boot=500,
    n_grid=500,
    n_bins=12,
    min_bin_n=10,
    random_state=123,
):
    rng = np.random.default_rng(random_state)

    sub = df_in[[x_col, y_col, cluster_col]].dropna().copy()
    sub = sub[sub[y_col].isin(class_order)].copy()

    if sub.empty:
        print(f"No data for {x_col}")
        return None

    print(f"\nPredictor: {x_col}")
    print(f"N = {len(sub)}")
    print(sub[y_col].value_counts().reindex(class_order))

    # ----------------------------
    # Fit original model
    # ----------------------------
    X = sub[[x_col]].to_numpy()
    y = sub[y_col].to_numpy()

    model = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=3000,
    )

    model.fit(X, y)

    x_min, x_max = np.nanpercentile(sub[x_col], [1, 99])
    x_grid = np.linspace(x_min, x_max, n_grid)

    probs_hat = predict_proba_fixed_order(model, x_grid, class_order)

    prob_df = pd.DataFrame(probs_hat, columns=class_order)
    prob_df[x_col] = x_grid

    prob_df["m_order"] = (
        prob_df["uncomfortable_side"]
        - prob_df["comfortable_side"]
    )

    threshold_equal = crossing_x(
        x_grid,
        prob_df["uncomfortable_side"] - prob_df["comfortable_side"],
    )

    threshold_u_1_3 = crossing_x(
        x_grid,
        prob_df["uncomfortable_side"] - (1 / 3),
    )

    threshold_c_1_3 = crossing_x(
        x_grid,
        prob_df["comfortable_side"] - (1 / 3),
    )

    # ----------------------------
    # Bootstrap
    # ----------------------------
    boot_probs = []
    boot_m = []
    boot_threshold_equal = []
    boot_threshold_u_1_3 = []
    boot_threshold_c_1_3 = []

    clusters = sub[cluster_col].dropna().unique()

    attempts = 0

    while len(boot_probs) < n_boot and attempts < n_boot * 10:
        attempts += 1

        sampled_clusters = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )

        boot_parts = []
        for cl in sampled_clusters:
            boot_parts.append(sub[sub[cluster_col] == cl])

        boot = pd.concat(boot_parts, axis=0)

        # Need all classes present
        if boot[y_col].nunique() < len(class_order):
            continue

        Xb = boot[[x_col]].to_numpy()
        yb = boot[y_col].to_numpy()

        try:
            mb = LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=3000,
            )
            mb.fit(Xb, yb)

            pb = predict_proba_fixed_order(mb, x_grid, class_order)

            m_b = pb[:, class_order.index("uncomfortable_side")] - pb[:, class_order.index("comfortable_side")]

            boot_probs.append(pb)
            boot_m.append(m_b)

            boot_threshold_equal.append(
                crossing_x(x_grid, m_b)
            )

            boot_threshold_u_1_3.append(
                crossing_x(
                    x_grid,
                    pb[:, class_order.index("uncomfortable_side")] - (1 / 3),
                )
            )

            boot_threshold_c_1_3.append(
                crossing_x(
                    x_grid,
                    pb[:, class_order.index("comfortable_side")] - (1 / 3),
                )
            )

        except Exception:
            continue

    boot_probs = np.asarray(boot_probs)
    boot_m = np.asarray(boot_m)

    boot_threshold_equal = np.asarray(boot_threshold_equal)
    boot_threshold_u_1_3 = np.asarray(boot_threshold_u_1_3)
    boot_threshold_c_1_3 = np.asarray(boot_threshold_c_1_3)

    print(f"Bootstrap fits used: {len(boot_probs)} / {n_boot}")

    # Probability CI
    prob_low = np.nanpercentile(boot_probs, 2.5, axis=0)
    prob_high = np.nanpercentile(boot_probs, 97.5, axis=0)

    # Order parameter CI
    m_low = np.nanpercentile(boot_m, 2.5, axis=0)
    m_high = np.nanpercentile(boot_m, 97.5, axis=0)

    # Threshold CIs
    th_equal_ci = np.nanpercentile(boot_threshold_equal, [2.5, 50, 97.5])
    th_u_1_3_ci = np.nanpercentile(boot_threshold_u_1_3, [2.5, 50, 97.5])
    th_c_1_3_ci = np.nanpercentile(boot_threshold_c_1_3, [2.5, 50, 97.5])

    print("\nThreshold estimates with bootstrap 95% CI:")
    print(
        f"P(U)=P(C): "
        f"{threshold_equal:.3f} "
        f"[{th_equal_ci[0]:.3f}, {th_equal_ci[2]:.3f}]"
    )
    print(
        f"P(U)=1/3: "
        f"{threshold_u_1_3:.3f} "
        f"[{th_u_1_3_ci[0]:.3f}, {th_u_1_3_ci[2]:.3f}]"
    )
    print(
        f"P(C)=1/3: "
        f"{threshold_c_1_3:.3f} "
        f"[{th_c_1_3_ci[0]:.3f}, {th_c_1_3_ci[2]:.3f}]"
    )

    # ----------------------------
    # Empirical binned probabilities with Wilson CI
    # ----------------------------
    sub["x_bin"] = pd.qcut(
        sub[x_col],
        q=n_bins,
        duplicates="drop",
    )

    rows = []

    for b, g in sub.groupby("x_bin", observed=False):
        n = len(g)

        if n < min_bin_n:
            continue

        row = {
            "x_mean": g[x_col].mean(),
            "x_median": g[x_col].median(),
            "n": n,
        }

        for cls in class_order:
            k = (g[y_col] == cls).sum()
            p = k / n
            lo, hi = wilson_ci(k, n)

            row[f"{cls}_p"] = p
            row[f"{cls}_lo"] = lo
            row[f"{cls}_hi"] = hi
            row[f"{cls}_k"] = k

        rows.append(row)

    bin_df = pd.DataFrame(rows)

    # ----------------------------
    # Plot 1: probabilities with bootstrap bands
    # ----------------------------
    plt.figure(figsize=(9, 5))

    for i, cls in enumerate(class_order):
        plt.fill_between(
            x_grid,
            prob_low[:, i],
            prob_high[:, i],
            alpha=0.18,
        )

        plt.plot(
            x_grid,
            probs_hat[:, i],
            linewidth=2,
            label=f"P({cls})",
        )

        if not bin_df.empty:
            y_bin = bin_df[f"{cls}_p"].values
            yerr = [
                y_bin - bin_df[f"{cls}_lo"].values,
                bin_df[f"{cls}_hi"].values - y_bin,
            ]

            plt.errorbar(
                bin_df["x_mean"],
                y_bin,
                yerr=yerr,
                fmt="o",
                capsize=3,
                alpha=0.65,
            )

    plt.axhline(1 / 3, linestyle="--", alpha=0.5, label="1/3")

    plt.axvline(
        threshold_equal,
        linestyle="--",
        alpha=0.8,
        label=f"P(U)=P(C): {threshold_equal:.2f}",
    )

    plt.axvspan(
        th_equal_ci[0],
        th_equal_ci[2],
        alpha=0.12,
        label="95% CI threshold",
    )

    plt.axvline(
        threshold_u_1_3,
        linestyle=":",
        alpha=0.8,
        label=f"P(U)=1/3: {threshold_u_1_3:.2f}",
    )

    plt.ylim(-0.02, 1.02)
    plt.xlabel(x_col)
    plt.ylabel("Predicted probability")
    plt.title(
        f"Comfort-side probabilities vs {x_col}\n"
        "Bands = walk-level bootstrap 95% CI"
    )
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # Plot 2: order parameter with bootstrap band
    # ----------------------------
    plt.figure(figsize=(9, 4.5))

    plt.fill_between(
        x_grid,
        m_low,
        m_high,
        alpha=0.22,
        label="95% bootstrap CI",
    )

    plt.plot(
        x_grid,
        prob_df["m_order"],
        linewidth=2,
        label="m = P(U) - P(C)",
    )

    plt.axhline(0, linestyle="--", alpha=0.7)
    plt.axvline(
        threshold_equal,
        linestyle="--",
        alpha=0.8,
        label=f"m=0 at {threshold_equal:.2f}",
    )

    plt.axvspan(
        th_equal_ci[0],
        th_equal_ci[2],
        alpha=0.12,
        label="95% CI threshold",
    )

    plt.xlabel(x_col)
    plt.ylabel("Order parameter m")
    plt.title(
        f"P(discomfort)-P(comfort) {x_col}\n"
        "Bands = walk-level bootstrap 95% CI"
    )
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # Plot 3: threshold bootstrap distribution
    # ----------------------------
    plt.figure(figsize=(8, 4.5))

    plt.hist(
        boot_threshold_equal,
        bins=30,
        alpha=0.75,
    )

    plt.axvline(threshold_equal, linestyle="--", label="Original estimate")
    plt.axvline(th_equal_ci[0], linestyle=":", label="2.5%")
    plt.axvline(th_equal_ci[2], linestyle=":", label="97.5%")

    plt.xlabel(x_col)
    plt.ylabel("Bootstrap count")
    plt.title("Bootstrap uncertainty of P(U)=P(C) threshold")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    result = {
        "predictor": x_col,
        "n": len(sub),
        "model": model,
        "prob_df": prob_df,
        "bin_df": bin_df,
        "x_grid": x_grid,
        "boot_probs": boot_probs,
        "boot_m": boot_m,
        "threshold_PU_eq_PC": threshold_equal,
        "threshold_PU_eq_PC_CI": th_equal_ci,
        "threshold_PU_eq_1_3": threshold_u_1_3,
        "threshold_PU_eq_1_3_CI": th_u_1_3_ci,
        "threshold_PC_eq_1_3": threshold_c_1_3,
        "threshold_PC_eq_1_3_CI": th_c_1_3_ci,
    }

    return result

In [ ]:
res_HDX = fit_comfort_side_model_with_error(
    df,
    x_col=hdx_col,
    cluster_col="walk",
    n_boot=500,
    n_bins=12,
)

In [ ]:
res_Tcorr = fit_comfort_side_model_with_error(
    df,
    x_col=temp_corr_col,
    cluster_col="walk",
    n_boot=500,
    n_bins=12,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


# ============================================================
# Helpers
# ============================================================

def crossing_x(x, y):
    """
    Finds x where y crosses 0.
    If no sign crossing exists, returns x where |y| is smallest.
    """
    x = np.asarray(x)
    y = np.asarray(y)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    if len(x) < 2:
        return np.nan

    sign_change = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]

    if len(sign_change) == 0:
        return x[np.nanargmin(np.abs(y))]

    i = sign_change[0]

    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]

    if y1 == y0:
        return x0

    return x0 - y0 * (x1 - x0) / (y1 - y0)


def make_multinomial_pipeline(numeric_cols, categorical_cols, categorical_categories=None):
    transformers = []

    if len(numeric_cols) > 0:
        transformers.append(
            ("num", StandardScaler(), numeric_cols)
        )

    if len(categorical_cols) > 0:
        if categorical_categories is None:
            encoder = OneHotEncoder(handle_unknown="ignore")
        else:
            encoder = OneHotEncoder(
                handle_unknown="ignore",
                categories=categorical_categories,
            )

        transformers.append(
            ("cat", encoder, categorical_cols)
        )

    preprocessor = ColumnTransformer(transformers=transformers)

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("logit", LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=4000,
            )),
        ]
    )

    return model


def predict_adjusted_curve(model, X_template, temp_col, t_grid, class_order):
    """
    For each temperature value, set all rows to that temperature,
    keep all other variables as observed, predict probabilities,
    then average over the observed covariate distribution.
    """
    curves = []

    for t in t_grid:
        X_temp = X_template.copy()
        X_temp[temp_col] = t

        probs = model.predict_proba(X_temp)

        prob_df = pd.DataFrame(
            probs,
            columns=model.named_steps["logit"].classes_,
        )

        for c in class_order:
            if c not in prob_df.columns:
                prob_df[c] = np.nan

        prob_df = prob_df[class_order]

        curves.append(prob_df.mean(axis=0).values)

    return np.asarray(curves)

In [ ]:
def fit_adjusted_temperature_curve_with_uncertainty(
    df_in,
    temp_col="<T-T_fixed+<T>>",
    y_col="comfort_3",
    numeric_cols=None,
    categorical_cols=None,
    cluster_col="walk",
    class_order=None,
    n_grid=300,
    n_boot=500,
    random_state=123,
    plot=True,
    title=None,
):
    """
    Multinomial logistic regression:
        comfort_3 ~ temperature + optional covariates

    The curve is plotted against temperature.
    Other variables are averaged over the observed data.

    Uncertainty is estimated by walk-level bootstrap.
    """

    if class_order is None:
        class_order = [
            "comfortable_side",
            "neutral",
            "uncomfortable_side",
        ]

    if numeric_cols is None:
        numeric_cols = [temp_col]

    if categorical_cols is None:
        categorical_cols = []

    # Make sure temp_col is included in numeric_cols
    if temp_col not in numeric_cols:
        numeric_cols = [temp_col] + numeric_cols

    needed_cols = [temp_col, y_col, cluster_col] + numeric_cols + categorical_cols
    needed_cols = list(dict.fromkeys(needed_cols))

    d = df_in[needed_cols].dropna().copy()
    d = d[d[y_col].isin(class_order)].copy()

    for c in numeric_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d.dropna(subset=numeric_cols).copy()

    print("\nModel:")
    print("Numeric:", numeric_cols)
    print("Categorical:", categorical_cols)
    print("N:", len(d))
    print(d[y_col].value_counts().reindex(class_order))

    X = d[numeric_cols + categorical_cols].copy()
    y = d[y_col].astype(str)

    # Fix categorical categories so bootstrap models use consistent encodings
    categorical_categories = None
    if len(categorical_cols) > 0:
        categorical_categories = [
            sorted(d[c].dropna().astype(str).unique())
            for c in categorical_cols
        ]

        for c in categorical_cols:
            X[c] = X[c].astype(str)

    # Temperature grid
    t_min, t_max = np.nanpercentile(d[temp_col], [1, 99])
    t_grid = np.linspace(t_min, t_max, n_grid)

    # --------------------------------------------------------
    # Original fit
    # --------------------------------------------------------
    model = make_multinomial_pipeline(
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        categorical_categories=categorical_categories,
    )

    model.fit(X, y)

    curve_hat = predict_adjusted_curve(
        model=model,
        X_template=X,
        temp_col=temp_col,
        t_grid=t_grid,
        class_order=class_order,
    )

    curve_df = pd.DataFrame(curve_hat, columns=class_order)
    curve_df[temp_col] = t_grid
    curve_df["m_order"] = (
        curve_df["uncomfortable_side"]
        - curve_df["comfortable_side"]
    )

    crossing_hat = crossing_x(
        t_grid,
        curve_df["m_order"],
    )

    # --------------------------------------------------------
    # Walk-level bootstrap
    # --------------------------------------------------------
    rng = np.random.default_rng(random_state)

    clusters = d[cluster_col].dropna().unique()

    boot_curves = []
    boot_crossings = []

    attempts = 0

    while len(boot_curves) < n_boot and attempts < n_boot * 10:
        attempts += 1

        sampled_clusters = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )

        boot_parts = []
        for cl in sampled_clusters:
            boot_parts.append(d[d[cluster_col] == cl])

        boot = pd.concat(boot_parts, axis=0).copy()

        if boot[y_col].nunique() < len(class_order):
            continue

        Xb = boot[numeric_cols + categorical_cols].copy()
        yb = boot[y_col].astype(str)

        for c in categorical_cols:
            Xb[c] = Xb[c].astype(str)

        try:
            mb = make_multinomial_pipeline(
                numeric_cols=numeric_cols,
                categorical_cols=categorical_cols,
                categorical_categories=categorical_categories,
            )

            mb.fit(Xb, yb)

            # IMPORTANT:
            # Predict over the original observed covariate distribution,
            # changing only temperature.
            cb = predict_adjusted_curve(
                model=mb,
                X_template=X,
                temp_col=temp_col,
                t_grid=t_grid,
                class_order=class_order,
            )

            m_b = (
                cb[:, class_order.index("uncomfortable_side")]
                - cb[:, class_order.index("comfortable_side")]
            )

            boot_curves.append(cb)
            boot_crossings.append(crossing_x(t_grid, m_b))

        except Exception:
            continue

    boot_curves = np.asarray(boot_curves)
    boot_crossings = np.asarray(boot_crossings)

    print(f"Bootstrap fits used: {len(boot_curves)} / {n_boot}")

    curve_low = np.nanpercentile(boot_curves, 2.5, axis=0)
    curve_med = np.nanpercentile(boot_curves, 50, axis=0)
    curve_high = np.nanpercentile(boot_curves, 97.5, axis=0)

    crossing_ci = np.nanpercentile(
        boot_crossings[np.isfinite(boot_crossings)],
        [2.5, 50, 97.5],
    )

    print("\nP(U)=P(C) crossing:")
    print(
        f"Original: {crossing_hat:.3f} "
        f"[{crossing_ci[0]:.3f}, {crossing_ci[2]:.3f}]"
    )
    print(f"Bootstrap median: {crossing_ci[1]:.3f}")

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    if plot:
        if title is None:
            title = "Adjusted comfort probabilities vs temperature"

        plt.figure(figsize=(9, 5))

        for i, cls in enumerate(class_order):
            plt.fill_between(
                t_grid,
                curve_low[:, i],
                curve_high[:, i],
                alpha=0.18,
            )

            plt.plot(
                t_grid,
                curve_hat[:, i],
                linewidth=2,
                label=f"P({cls})",
            )

        plt.axhline(
            1 / len(class_order),
            linestyle="--",
            alpha=0.6,
            label=f"1/{len(class_order)}",
        )

        plt.axvline(
            crossing_hat,
            linestyle=":",
            alpha=0.9,
            label=f"P(U)=P(C) ≈ {crossing_hat:.2f}",
        )

        plt.axvspan(
            crossing_ci[0],
            crossing_ci[2],
            alpha=0.12,
            label="Bootstrap crossing 95% CI",
        )

        plt.xlabel(temp_col)
        plt.ylabel("Adjusted predicted probability")
        plt.title(
            title + "\n"
            "Bands = walk-level bootstrap 95% CI"
        )
        plt.ylim(-0.02, 1.02)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        # Order parameter plot
        m_hat = (
            curve_hat[:, class_order.index("uncomfortable_side")]
            - curve_hat[:, class_order.index("comfortable_side")]
        )

        boot_m = (
            boot_curves[:, :, class_order.index("uncomfortable_side")]
            - boot_curves[:, :, class_order.index("comfortable_side")]
        )

        m_low = np.nanpercentile(boot_m, 2.5, axis=0)
        m_high = np.nanpercentile(boot_m, 97.5, axis=0)

        plt.figure(figsize=(9, 4.5))

        plt.fill_between(
            t_grid,
            m_low,
            m_high,
            alpha=0.22,
            label="95% bootstrap CI",
        )

        plt.plot(
            t_grid,
            m_hat,
            linewidth=2,
            label="m = P(U) - P(C)",
        )

        plt.axhline(0, linestyle="--", alpha=0.7)
        plt.axvline(
            crossing_hat,
            linestyle=":",
            alpha=0.9,
            label=f"m=0 ≈ {crossing_hat:.2f}",
        )
        plt.axvspan(
            crossing_ci[0],
            crossing_ci[2],
            alpha=0.12,
            label="Bootstrap crossing 95% CI",
        )

        plt.xlabel(temp_col)
        plt.ylabel("Order parameter")
        plt.title(
            "Adjusted order parameter vs temperature\n"
            "Bands = walk-level bootstrap 95% CI"
        )
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "model": model,
        "curve_df": curve_df,
        "t_grid": t_grid,
        "curve_hat": curve_hat,
        "curve_low": curve_low,
        "curve_med": curve_med,
        "curve_high": curve_high,
        "boot_curves": boot_curves,
        "crossing": crossing_hat,
        "crossing_CI": crossing_ci,
        "boot_crossings": boot_crossings,
        "data_used": d,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
    }

In [ ]:
# res_T_only = fit_adjusted_temperature_curve_with_uncertainty(
#     df,
#     temp_col="<T-T_fixed+<T>>",
#     y_col="comfort_3",
#     numeric_cols=["<T-T_fixed+<T>>"],
#     categorical_cols=[],
#     cluster_col="walk",
#     n_boot=500,
#     title="Temperature-only comfort probabilities",
# )

In [ ]:
res_T_only = fit_adjusted_temperature_curve_with_uncertainty(
    df,
    temp_col="<T-T_fixed+<T>>",
    y_col="comfort_3",
    numeric_cols=["<T-T_fixed+<T>>"],
    categorical_cols=["gender"],
    cluster_col="walk",
    n_boot=500,
    title="Temperature-only comfort probabilities",
)

In [ ]:
# res_T_sun_wind_walk = fit_adjusted_temperature_curve_with_uncertainty(
#     df,
#     temp_col="<T-T_fixed+<T>>",
#     y_col="comfort_3",
#     numeric_cols=["<T-T_fixed+<T>>"],
#     categorical_cols=["sun", "wind", "walk"],
#     cluster_col="walk",
#     n_boot=500,
#     title="Temperature effect adjusted for sun, wind, and walk",
# )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def wilson_ci(k, n, confidence=0.95):
    if n == 0:
        return np.nan, np.nan

    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    phat = k / n

    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (
        z
        * np.sqrt((phat * (1 - phat) / n) + (z**2 / (4 * n**2)))
        / denom
    )

    return center - half, center + half


def empirical_comfort_probabilities(
    df_in,
    x_col,
    y_col="comfort_3",
    class_order=None,
    n_bins=12,
    min_bin_n=20,
    binning="quantile",  # "quantile" or "equal_width"
    plot=True,
    shade_crossing=True,
):
    if class_order is None:
        class_order = [
            "comfortable_side",
            "uncomfortable_side",
        ]

    sub = df_in[[x_col, y_col]].dropna().copy()
    sub = sub[sub[y_col].isin(class_order)].copy()

    if binning == "quantile":
        sub["x_bin"] = pd.qcut(
            sub[x_col],
            q=n_bins,
            duplicates="drop",
        )
    elif binning == "equal_width":
        sub["x_bin"] = pd.cut(
            sub[x_col],
            bins=n_bins,
            include_lowest=True,
        )
    else:
        raise ValueError("binning must be 'quantile' or 'equal_width'")

    rows = []

    for b, g in sub.groupby("x_bin", observed=False):
        n = len(g)

        if n < min_bin_n:
            continue

        row = {
            "bin": b,
            "bin_left": b.left if hasattr(b, "left") else np.nan,
            "bin_right": b.right if hasattr(b, "right") else np.nan,
            "x_mean": g[x_col].mean(),
            "x_median": g[x_col].median(),
            "n": n,
        }

        for cls in class_order:
            k = (g[y_col] == cls).sum()
            p = k / n
            lo, hi = wilson_ci(k, n)

            row[f"{cls}_count"] = k
            row[f"{cls}_p"] = p
            row[f"{cls}_ci_low"] = lo
            row[f"{cls}_ci_high"] = hi

        rows.append(row)

    prob_bins = pd.DataFrame(rows).sort_values("x_mean").reset_index(drop=True)

    if prob_bins.empty:
        print("No valid bins after min_bin_n filtering.")
        return prob_bins, np.nan, None

    # --------------------------------------------------
    # Empirical crossing P(U)=P(C)
    # --------------------------------------------------
    diff = (
        prob_bins["uncomfortable_side_p"]
        - prob_bins["comfortable_side_p"]
    )

    crossing = np.nan
    crossing_interval = None

    sign_change = np.where(
        np.sign(diff.values[:-1]) != np.sign(diff.values[1:])
    )[0]

    if len(sign_change) > 0:
        i = sign_change[0]

        x0 = prob_bins["x_mean"].iloc[i]
        x1 = prob_bins["x_mean"].iloc[i + 1]
        y0 = diff.iloc[i]
        y1 = diff.iloc[i + 1]

        crossing = x0 - y0 * (x1 - x0) / (y1 - y0)

        # The empirical crossing must lie between these two bin centers
        crossing_interval = (min(x0, x1), max(x0, x1))

    else:
        idx = diff.abs().idxmin()
        crossing = prob_bins.loc[idx, "x_mean"]

        # If no sign change, use the closest bin as approximate crossing area
        crossing_interval = (
            prob_bins.loc[idx, "bin_left"],
            prob_bins.loc[idx, "bin_right"],
        )

    # --------------------------------------------------
    # Confidence-overlap area
    # Bins where Wilson CIs of C and U overlap
    # --------------------------------------------------
    ci_overlap = (
        (
            prob_bins["comfortable_side_ci_low"]
            <= prob_bins["uncomfortable_side_ci_high"]
        )
        &
        (
            prob_bins["uncomfortable_side_ci_low"]
            <= prob_bins["comfortable_side_ci_high"]
        )
    )

    overlap_bins = prob_bins[ci_overlap].copy()

    if len(overlap_bins) > 0:
        ci_overlap_interval = (
            overlap_bins["bin_left"].min(),
            overlap_bins["bin_right"].max(),
        )
    else:
        ci_overlap_interval = None

    print("Empirical binned probabilities:")
    print(prob_bins.round(3))

    print(f"\nApprox empirical P(U)=P(C) crossing: {crossing:.3f}")

    if crossing_interval is not None:
        print(
            "Nominal crossing area between bin centers: "
            f"[{crossing_interval[0]:.3f}, {crossing_interval[1]:.3f}]"
        )

    if ci_overlap_interval is not None:
        print(
            "Wilson-CI overlap area: "
            f"[{ci_overlap_interval[0]:.3f}, {ci_overlap_interval[1]:.3f}]"
        )
    else:
        print("No Wilson-CI overlap area found.")

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------
    if plot:
        plt.figure(figsize=(9, 5))

        # Wider uncertainty area: where C and U confidence intervals overlap
        if shade_crossing and ci_overlap_interval is not None:
            plt.axvspan(
                ci_overlap_interval[0],
                ci_overlap_interval[1],
                alpha=0.12,
                label="Wilson-CI overlap area",
            )

        # Narrower nominal crossing area
        

        for cls in class_order:
            y = prob_bins[f"{cls}_p"].values
            yerr = [
                y - prob_bins[f"{cls}_ci_low"].values,
                prob_bins[f"{cls}_ci_high"].values - y,
            ]

            plt.errorbar(
                prob_bins["x_mean"],
                y,
                yerr=yerr,
                marker="o",
                capsize=4,
                label=cls,
            )

        # Since you only keep two classes, the neutral class is excluded.
        # Therefore the equal-probability reference is 0.5, not 1/3.
        if len(class_order) == 2:
            reference = 0.5
            reference_label = "0.5"
        else:
            reference = 1 / len(class_order)
            reference_label = f"1/{len(class_order)}"

        plt.axhline(
            reference,
            linestyle="--",
            alpha=0.6,
            label=reference_label,
        )

        if np.isfinite(crossing):
            plt.axvline(
                crossing,
                linestyle=":",
                alpha=0.9,
                label=f"Empirical P(U)=P(C) ≈ {crossing:.2f}",
            )

        plt.xlabel(x_col)
        plt.ylabel("Empirical probability")
        plt.title(f"Empirical comfort probabilities by {x_col} bins")
        plt.ylim(-0.02, 1.02)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    crossing_info = {
        "crossing": crossing,
        "crossing_interval": crossing_interval,
        "ci_overlap_interval": ci_overlap_interval,
        "overlap_bins": overlap_bins,
    }

    return prob_bins, crossing, crossing_info

In [ ]:
prob_T, crossing_T, crossing_info_T = empirical_comfort_probabilities(
    df,
    x_col="<T-T_fixed+<T>>",
    y_col="comfort_3",
    n_bins=12,
    min_bin_n=20,
    binning="quantile",
)

In [ ]:
prob_HDX, crossing_HDX, crossing_info_HDX = empirical_comfort_probabilities(
    df,
    x_col="<HDX-HDX_fixed+<HDX>>",
    y_col="comfort_3",
    n_bins=12,
    min_bin_n=20,
    binning="quantile",
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# =========================
# CONFIG
# =========================
temp_corr_col = "<T-T_fixed+<T>>"
hdx_col = "<HDX-HDX_fixed+<HDX>>"
comfort_col = "thermal_comfort"
sun_col = "sun"
wind_col = "wind"

# =========================
# PREPARE DATA
# =========================
df = df_votes.copy()
df[temp_corr_col] = pd.to_numeric(df[temp_corr_col], errors="coerce")
df[hdx_col] = pd.to_numeric(df[hdx_col], errors="coerce")

def comfort_side(cat):
    if cat in ["Very comfortable", "Comfortable", "Slightly comfortable"]:
        return "comfortable_side"
    elif cat == "Neutral":
        return "neutral"
    elif cat in ["Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"]:
        return "uncomfortable_side"
    return np.nan

df["comfort_3"] = df[comfort_col].map(comfort_side)

# simple environmental groupings
def sun_group(val):
    if val == "In full sun":
        return "sun"
    elif val == "In full shade":
        return "shade"
    return np.nan

def wind_group(val):
    if val == "It's not windy":
        return "no_wind"
    elif val in ["A strong wind", "A moderate wind", "A light wind", "A very light wind"]:
        return "wind"
    return np.nan

df["sun_group"] = df[sun_col].map(sun_group)
df["wind_group"] = df[wind_col].map(wind_group)
df["sun_wind_group"] = df["sun_group"].fillna("other") + " + " + df["wind_group"].fillna("other")

In [ ]:
def fit_comfort_order_parameter(
    df_in,
    x_col,
    y_col="comfort_3",
    n_grid=500,
    plot=True,
    title=None,
):
    """
    Fits a multinomial logistic model for:
      comfortable_side / neutral / uncomfortable_side

    Defines the order-like parameter:
      m(x) = P(uncomfortable_side | x) - P(comfortable_side | x)

    Returns the threshold where m(x)=0, i.e. P(U)=P(C).
    """
    sub = df_in[[x_col, y_col]].dropna().copy()

    class_order = ["comfortable_side", "neutral", "uncomfortable_side"]
    sub = sub[sub[y_col].isin(class_order)].copy()

    if sub.empty or sub[y_col].nunique() < 2:
        print(f"Not enough data for {x_col}")
        return None

    X = sub[[x_col]].to_numpy()
    y = pd.Categorical(sub[y_col], categories=class_order, ordered=True)

    model = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=2000,
    )
    model.fit(X, y)

    x_min, x_max = np.nanpercentile(sub[x_col], [1, 99])
    x_grid = np.linspace(x_min, x_max, n_grid).reshape(-1, 1)

    probs = model.predict_proba(x_grid)
    prob_df = pd.DataFrame(probs, columns=model.classes_)
    prob_df[x_col] = x_grid[:, 0]

    for c in class_order:
        if c not in prob_df.columns:
            prob_df[c] = np.nan
    prob_df = prob_df[[x_col] + class_order]

    # first order-like parameter only
    prob_df["m_order"] = prob_df["uncomfortable_side"] - prob_df["comfortable_side"]

    # threshold where m(x)=0  <=>  P(U)=P(C)
    idx_thr = np.argmin(np.abs(prob_df["m_order"]))
    threshold = prob_df.iloc[idx_thr][x_col]

    result = {
        "predictor": x_col,
        "n": len(sub),
        "threshold_m_eq_0": threshold,
        "model": model,
        "prob_df": prob_df,
    }

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

        # probabilities
        axes[0].plot(prob_df[x_col], prob_df["comfortable_side"], label="P(comfortable side)")
        axes[0].plot(prob_df[x_col], prob_df["neutral"], label="P(neutral)")
        axes[0].plot(prob_df[x_col], prob_df["uncomfortable_side"], label="P(uncomfortable side)")
        axes[0].axvline(threshold, linestyle="--", alpha=0.8, label=f"P(U)=P(C): {threshold:.2f}")
        axes[0].set_ylim(-0.02, 1.02)
        axes[0].set_xlabel(x_col)
        axes[0].set_ylabel("Predicted probability")
        axes[0].set_title(title if title else f"Comfort-side probabilities vs {x_col}")
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        # order parameter
        axes[1].plot(prob_df[x_col], prob_df["m_order"], label="m(x) = P(U) - P(C)")
        axes[1].axhline(0, linestyle="--", alpha=0.7)
        axes[1].axvline(threshold, linestyle="--", alpha=0.8, label=f"m=0 at {threshold:.2f}")
        axes[1].set_xlabel(x_col)
        axes[1].set_ylabel("Order-like parameter")
        axes[1].set_title(title if title else f"Order-like parameter vs {x_col}")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

    return result

In [ ]:
results = []

for predictor in [temp_corr_col, hdx_col]:
    res = fit_comfort_order_parameter(
        df,
        x_col=predictor,
        y_col="comfort_3",
        plot=True,
        title=f"All data | {predictor}",
    )
    if res is not None:
        results.append({
            "subset": "all",
            "predictor": predictor,
            "n": res["n"],
            "threshold_m_eq_0": res["threshold_m_eq_0"],
        })

results_df = pd.DataFrame(results)
print("\n=== ALL DATA THRESHOLDS ===")
print(results_df)

In [ ]:
sun_results = []

for predictor in [temp_corr_col, hdx_col]:
    for grp in ["sun", "shade"]:
        sub = df[df["sun_group"] == grp].copy()
        res = fit_comfort_order_parameter(
            sub,
            x_col=predictor,
            y_col="comfort_3",
            plot=True,
            title=f"sun_group = {grp} | {predictor}",
        )
        if res is not None:
            sun_results.append({
                "subset": f"sun_group={grp}",
                "predictor": predictor,
                "n": res["n"],
                "threshold_m_eq_0": res["threshold_m_eq_0"],
            })

sun_results_df = pd.DataFrame(sun_results)
print("\n=== SUN VS SHADE THRESHOLDS ===")
print(sun_results_df)

In [ ]:
wind_results = []

for predictor in [temp_corr_col, hdx_col]:
    for grp in ["wind", "no_wind"]:
        sub = df[df["wind_group"] == grp].copy()
        res = fit_comfort_order_parameter(
            sub,
            x_col=predictor,
            y_col="comfort_3",
            plot=True,
            title=f"wind_group = {grp} | {predictor}",
        )
        if res is not None:
            wind_results.append({
                "subset": f"wind_group={grp}",
                "predictor": predictor,
                "n": res["n"],
                "threshold_m_eq_0": res["threshold_m_eq_0"],
            })

wind_results_df = pd.DataFrame(wind_results)
print("\n=== WIND VS NO-WIND THRESHOLDS ===")
print(wind_results_df)

In [ ]:
combo_results = []

combo_levels = [
    "sun + wind",
    "sun + no_wind",
    "shade + wind",
    "shade + no_wind",
]

for predictor in [temp_corr_col, hdx_col]:
    for grp in combo_levels:
        sub = df[df["sun_wind_group"] == grp].copy()
        res = fit_comfort_order_parameter(
            sub,
            x_col=predictor,
            y_col="comfort_3",
            plot=True,
            title=f"{grp} | {predictor}",
        )
        if res is not None:
            combo_results.append({
                "subset": grp,
                "predictor": predictor,
                "n": res["n"],
                "threshold_m_eq_0": res["threshold_m_eq_0"],
            })

combo_results_df = pd.DataFrame(combo_results)
print("\n=== COMBINED SUN/WIND THRESHOLDS ===")
print(combo_results_df)

# Gènere

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# =========================
# CONFIG
# =========================
temp_corr_col = "<T-T_fixed+<T>>"
hdx_col = "<HDX-HDX_fixed+<HDX>>"

comfort_col = "thermal_comfort"
sensation_col = "thermal_sensation"
gender_col = "gender"

valid_genders = ["Woman", "Man"]

# =========================
# PREPARE DATA
# =========================
df = df_votes.copy()
df = df[df[gender_col].isin(valid_genders)].copy()

df[temp_corr_col] = pd.to_numeric(df[temp_corr_col], errors="coerce")
df[hdx_col] = pd.to_numeric(df[hdx_col], errors="coerce")

# ----- comfort: 3 groups -----
def comfort_side(cat):
    if cat in ["Very comfortable", "Comfortable", "Slightly comfortable"]:
        return "comfortable_side"
    elif cat == "Neutral":
        return "neutral"
    elif cat in ["Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"]:
        return "uncomfortable_side"
    return np.nan

df["comfort_3"] = df[comfort_col].map(comfort_side)

# ----- sensation: 3 groups -----
def sensation_side(cat):
    if cat in ["Cool", "Slightly cool"]:
        return "cool_side"
    elif cat in ["Neutral", "Slightly warm", "Warm"]:
        return "middle"
    elif cat in ["Hot", "Very hot"]:
        return "hot_side"
    return np.nan

df["sensation_3"] = df[sensation_col].map(sensation_side)



def fit_three_group_threshold(
    df_in,
    x_col,
    y_col,
    class_order,
    left_class,
    right_class,
    label="",
    n_grid=500,
    plot=True,
):
    """
    Fits multinomial logistic regression for 3 groups and defines:

        m(x) = P(right_class | x) - P(left_class | x)

    Threshold = x where m(x)=0.
    """
    sub = df_in[[x_col, y_col]].dropna().copy()
    sub = sub[sub[y_col].isin(class_order)].copy()

    if sub.empty or sub[y_col].nunique() < 2:
        print(f"{label} | {x_col}: not enough data")
        return None

    X = sub[[x_col]].to_numpy()
    y = pd.Categorical(sub[y_col], categories=class_order, ordered=True)

    model = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=3000,
    )
    model.fit(X, y)

    x_min, x_max = np.nanpercentile(sub[x_col], [1, 99])
    x_grid = np.linspace(x_min, x_max, n_grid).reshape(-1, 1)

    probs = model.predict_proba(x_grid)
    prob_df = pd.DataFrame(probs, columns=model.classes_)
    prob_df[x_col] = x_grid[:, 0]

    # ensure all classes appear as columns
    for c in class_order:
        if c not in prob_df.columns:
            prob_df[c] = np.nan
    prob_df = prob_df[[x_col] + class_order]

    # signed order-like parameter
    prob_df["m_order"] = prob_df[right_class] - prob_df[left_class]

    # threshold where m(x)=0
    idx_thr = np.argmin(np.abs(prob_df["m_order"]))
    threshold = prob_df.iloc[idx_thr][x_col]

    result = {
        "predictor": x_col,
        "n": len(sub),
        "threshold_m_eq_0": threshold,
        "model": model,
        "prob_df": prob_df,
        "left_class": left_class,
        "right_class": right_class,
    }

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

        # probability curves
        for c in class_order:
            axes[0].plot(prob_df[x_col], prob_df[c], label=f"P({c})")

        axes[0].axvline(threshold, linestyle="--", alpha=0.8, label=f"m=0 at {threshold:.2f}")
        axes[0].set_ylim(-0.02, 1.02)
        axes[0].set_xlabel(x_col)
        axes[0].set_ylabel("Predicted probability")
        axes[0].set_title(f"{label} | probabilities vs {x_col}")
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        # order-like parameter
        axes[1].plot(prob_df[x_col], prob_df["m_order"], label=f"m = P({right_class}) - P({left_class})")
        axes[1].axhline(0, linestyle="--", alpha=0.7)
        axes[1].axvline(threshold, linestyle="--", alpha=0.8, label=f"m=0 at {threshold:.2f}")
        axes[1].set_xlabel(x_col)
        axes[1].set_ylabel("Order-like parameter")
        axes[1].set_title(f"{label} | order-like parameter vs {x_col}")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

    return result

def run_comfort_thresholds_by_gender(df_in, predictors, gender_col="gender", plot=True):
    class_order = ["comfortable_side", "neutral", "uncomfortable_side"]
    left_class = "comfortable_side"
    right_class = "uncomfortable_side"

    rows = []

    groups = [("All", df_in)] + [(g, df_in[df_in[gender_col] == g].copy()) for g in ["Woman", "Man"]]

    for group_name, sub in groups:
        for x_col in predictors:
            res = fit_three_group_threshold(
                df_in=sub,
                x_col=x_col,
                y_col="comfort_3",
                class_order=class_order,
                left_class=left_class,
                right_class=right_class,
                label=f"Comfort | {group_name}",
                plot=plot,
            )
            if res is not None:
                rows.append({
                    "domain": "comfort",
                    "group": group_name,
                    "predictor": x_col,
                    "n": res["n"],
                    "threshold_m_eq_0": res["threshold_m_eq_0"],
                })

    return pd.DataFrame(rows)

comfort_thresholds_df = run_comfort_thresholds_by_gender(
    df,
    predictors=[temp_corr_col, hdx_col],
    gender_col=gender_col,
    plot=True,
)

print("\n=== COMFORT THRESHOLDS BY GENDER ===")
print(comfort_thresholds_df)

In [ ]:
def run_sensation_thresholds_by_gender(df_in, predictors, gender_col="gender", plot=True):
    class_order = ["cool_side", "middle", "hot_side"]
    left_class = "cool_side"
    right_class = "hot_side"

    rows = []

    groups = [("All", df_in)] + [(g, df_in[df_in[gender_col] == g].copy()) for g in ["Woman", "Man"]]

    for group_name, sub in groups:
        for x_col in predictors:
            res = fit_three_group_threshold(
                df_in=sub,
                x_col=x_col,
                y_col="sensation_3",
                class_order=class_order,
                left_class=left_class,
                right_class=right_class,
                label=f"Sensation | {group_name}",
                plot=plot,
            )
            if res is not None:
                rows.append({
                    "domain": "sensation",
                    "group": group_name,
                    "predictor": x_col,
                    "n": res["n"],
                    "threshold_m_eq_0": res["threshold_m_eq_0"],
                })

    return pd.DataFrame(rows)

sensation_thresholds_df = run_sensation_thresholds_by_gender(
    df,
    predictors=[temp_corr_col, hdx_col],
    gender_col=gender_col,
    plot=True,
)

print("\n=== SENSATION THRESHOLDS BY GENDER ===")
print(sensation_thresholds_df)